# Goal
Characterize the different kinds of latents we see in the circuit and try to form methods to detect / classify them in an unsupervised fashion 

# Setup

In [1]:
import argparse
import torch
import json
import torch.nn.functional as F
from tqdm import tqdm
import pickle
from typing import Sequence, Tuple, Optional, List
from scipy.stats import mannwhitneyu
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact, rankdata
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from scipy.signal import savgol_filter, find_peaks
import ruptures as rpt
from sklearn.mixture import GaussianMixture
import time
import os

import sys
sys.path.append('../../')
sys.path.append('../../plm_circuits')

# Import utility functions
from helpers.utils import (
    clear_memory,
    load_esm,
    load_sae_prot,
    mask_flanks_segment,
    patching_metric,
    cleanup_cuda,
    set_seed
)

# Import attribution functions
from attribution import (
    integrated_gradients_sae,
    topk_sae_err_pt
)

# Import hook classes
from hook_manager import SAEHookProt

from data.protein_params import sse_dict, fl_dict, protein_name, protein2pdb
from data.feature_clusters_hypotheses import feature_clusters_MetXA, feature_clusters_Top2

# Additional imports
import json
from functools import partial
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import collections
from typing import Dict, List, Tuple, Optional

# Lazy caches to avoid heavy loads on import
_ESM_CACHE = None  # type: ignore[var-annotated]
_SAE_CACHE: Dict[int, object] = {}


In IPython
Set autoreload
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
In IPython
Set autoreload


In [2]:
all_acts_mean = torch.load("/project/pi_annagreen_umass_edu/bryn/plm_circuit_enrichment/data/summarized_acts_length_normalized.pt", weights_only=True) 
all_acts_max = torch.load("/project/pi_annagreen_umass_edu/bryn/plm_circuit_enrichment/data/summarized_acts_max.pt", weights_only=True)
all_acts_topq = torch.load("data/summarized_acts_top_q.pt", weights_only=True)

all_properties = torch.load("metadata/ptn_fam_tensor_nonzero.pt", weights_only=True)

with open("metadata/list_of_desired_latents.pkl", 'rb') as f:
    subset_list = list(set(pickle.load(f)))
    subset_list.sort()

interpro_annotations_nonzero = pd.read_csv("metadata/interpro_entry_list_mapping_nonzero.csv")

all_acts_mean = all_acts_mean[:, subset_list]
all_acts_max = all_acts_max[:, subset_list]
all_acts_topq = all_acts_topq[:, subset_list]

print(f"Number of latents: {len(subset_list)}")


with open("/work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/results/layer_latent_dicts/layer_latent_dict_2PKEA_0.70.json", "r") as f:
    pkea_latents = json.load(f)

with open("/work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/results/layer_latent_dicts/layer_latent_dict_MetXA_0.70.json", 'r') as file:
    metx_latents = json.load(file)

with open("/work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/results/layer_latent_dicts/layer_latent_dict_Top2_0.70.json", 'r') as file:
    top2_latents = json.load(file)


# only working on metxa and top2 for now
offsets = {k: i*4096 for i,k in enumerate(metx_latents.keys())}
full_indices_metx = [j+offsets[layer_id] for layer_id in metx_latents.keys() for j in metx_latents[layer_id]]
full_indices_top2 = [j+offsets[layer_id] for layer_id in top2_latents.keys() for j in top2_latents[layer_id]]
# full_indices_pkea = [j+offsets[layer_id] for layer_id in pkea_latents.keys() for j in pkea_latents[layer_id]]
latent_ids_both_proteins = list(set(full_indices_top2+full_indices_metx))
# latent_ids_both_proteins = list(set(full_indices_pkea))
latent_ids_both_proteins.sort()
print(f"Number of latent ids both proteins: {len(latent_ids_both_proteins)}, equal latents? {subset_list == latent_ids_both_proteins}")


Number of latents: 383
Number of latent ids both proteins: 383, equal latents? True


## Helper functions

In [3]:
def to_flattened_id(layer_id: int, lat_id: int) -> int:
    if layer_id % 4 != 0:
        raise ValueError("layer_id must be a multiple of 4 (e.g., 4, 8, 12, ...)")
    if not (0 <= lat_id < 4096):
        raise ValueError("lat_id must be in [0, 4096)")
    return ((layer_id // 4) - 1) * 4096 + lat_id

def get_layer_and_latent(latent_i: int, flattened_ids: Sequence[int]) -> Tuple[int, int]:
    """
    Given a latent index and a list/array of flattened IDs,
    return (layer_id, latent_ind).

    Assumes:
    - flattened_id -> layer: ((flattened_id // 4096) + 1) * 4
    - flattened_id -> latent_ind: flattened_id % 4096
    """
    if latent_i < 0 or latent_i >= len(flattened_ids):
        raise IndexError("latent_i out of range for flattened_ids")
    cur_flattened = int(flattened_ids[latent_i])
    layer_id = ((cur_flattened // 4096) + 1) * 4
    latent_ind = cur_flattened % 4096
    return layer_id, latent_ind


### activation curve

In [4]:
out_dir = Path("../results/act_sorted_max/")
out_dir.mkdir(parents=True, exist_ok=True)
def save_activation_curve(layer_i, latent_ind, tau=0.99, symlog=False, out_dir=out_dir):
    flattened_id = to_flattened_id(int(layer_i), int(latent_ind))
    latent_i = subset_list.index(flattened_id)

    v = to_numpy(all_acts)[:, latent_i].astype(float)
    order = np.argsort(v)[::-1]
    s = v[order]
    N = s.size

    k_top = max(1, int(np.ceil((1 - tau) * N)))

    if N > 1 and (s.max() - s.min()) > 0:
        x = np.linspace(0.0, 1.0, N)
        y = (s - s.min()) / (s.max() - s.min() + 1e-12)
        line = 1.0 - x
        knee_idx = int(np.argmax(y - line))
    else:
        knee_idx = 0

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.arange(1, N + 1), s, lw=1.2)
    ax.axvline(k_top, color="crimson", ls="--", alpha=0.7, label=f"top {100*(1-tau):.1f}%")
    ax.axvline(knee_idx + 1, color="teal", ls=":", alpha=0.7, label="knee")
    ax.set_xlabel("Proteins (sorted by latent activation, rank)")
    ax.set_ylabel("Activation")
    ax.set_xlim(1, N)
    if symlog:
        ax.set_yscale("symlog")
    ax.legend(frameon=False)
    ax.set_title(f"L{layer_i}-{latent_ind} (flat {flattened_id})")
    plt.tight_layout()

    out_path = out_dir / f"L{layer_i}_{latent_ind}_sorted_curve.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path.resolve()}")

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


def save_activation_curve_og(
    layer_i,
    latent_ind,
    tau=0.99,
    symlog=False,
    highlight_domain_idx=None,  # NEW: domain column index to highlight
    highlight_color="orange",
    highlight_alpha=0.8,
    highlight_marker_size=18,
    out_dir=None,
    all_acts=None,
):
    """
    Plot sorted activations for (layer_i, latent_ind).

    If highlight_domain_idx is provided, overlay points corresponding to proteins
    that belong to that domain (column in all_properties != 0), and report
    coverage at the current tau threshold.
    """
    flattened_id = to_flattened_id(int(layer_i), int(latent_ind))
    latent_i = subset_list.index(flattened_id)

    v = to_numpy(all_acts)[:, latent_i].astype(float)
    order = np.argsort(v)[::-1]
    s = v[order]
    N = s.size

    # Default output directory resolution
    if out_dir is None:
        out_dir_use = globals().get("out_dir", Path("./"))
    else:
        out_dir_use = out_dir
    if isinstance(out_dir_use, str):
        out_dir_use = Path(out_dir_use)
    out_dir_use.mkdir(parents=True, exist_ok=True)

    # Position of current top-tail threshold
    k_top = max(1, int(np.ceil((1 - float(tau)) * N)))

    # Knee heuristic (optional visual aid)
    if N > 1 and (s.max() - s.min()) > 0:
        x = np.linspace(0.0, 1.0, N)
        y = (s - s.min()) / (s.max() - s.min() + 1e-12)
        line = 1.0 - x
        knee_idx = int(np.argmax(y - line))
    else:
        knee_idx = 0

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.arange(1, N + 1), s, lw=1.2, color="#1f77b4")
    ax.axvline(k_top, color="crimson", ls="--", alpha=0.7, label=f"top {100*(1-float(tau)):.1f}%")
    ax.axvline(knee_idx + 1, color="teal", ls=":", alpha=0.7, label="knee")

    # Optional domain highlighting
    if highlight_domain_idx is not None:
        try:
            D = to_numpy(all_properties)
            col = int(highlight_domain_idx)
            if col < 0 or col >= D.shape[1]:
                raise IndexError(f"highlight_domain_idx {col} out of range [0, {D.shape[1]-1}]")
            in_mask = (D[:, col] != 0)
            in_mask_sorted = in_mask[order]
            idxs = np.flatnonzero(in_mask_sorted)

            # Coverage at current tau (what fraction of domain is within top k?)
            n_in = int(in_mask.sum())
            n_in_top = int(in_mask_sorted[:k_top].sum())
            coverage = (n_in_top / n_in) if n_in > 0 else 0.0

            # Try to resolve domain name for legend
            label_name = None
            try:
                df_ann = globals().get("interpro_annotations_nonzero", None)
                if df_ann is not None:
                    label_name = str(df_ann.iloc[col]["ENTRY_NAME"])  # may raise
            except Exception:
                label_name = None

            label = f"domain {col}"
            if label_name:
                label = f"{label_name} (idx {col})"
            label += f"; n={n_in}; cov@tau={coverage:.0%}"

            if idxs.size > 0:
                ax.scatter(
                    idxs + 1,
                    s[idxs],
                    s=highlight_marker_size,
                    color=highlight_color,
                    alpha=highlight_alpha,
                    edgecolors="none",
                    label=label,
                    zorder=3,
                )
            else:
                # Still add legend entry for clarity
                ax.scatter([], [], s=highlight_marker_size, color=highlight_color, alpha=highlight_alpha, edgecolors="none", label=label)
        except Exception as e:
            ax.scatter([], [], s=highlight_marker_size, color=highlight_color, alpha=highlight_alpha, edgecolors="none", label=f"domain {highlight_domain_idx} (error)")
            print(f"Highlight error for domain {highlight_domain_idx}: {e}")

    ax.set_xlabel("Proteins (sorted by latent activation, rank)")
    ax.set_ylabel("Activation")
    ax.set_xlim(1, N)
    if symlog:
        ax.set_yscale("symlog")
    ax.legend(frameon=False)
    ax.set_title(f"L{layer_i}-{latent_ind} (flat {flattened_id})")
    plt.tight_layout()

    out_path = out_dir_use / f"L{layer_i}_{latent_ind}_sorted_curve.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path.resolve()}")

def to_numpy(a):
    if torch is not None and isinstance(a, torch.Tensor):
        return a.detach().cpu().numpy()
    return np.asarray(a)

In [ ]:
save_activation_curve_og(8, 2677, highlight_domain_idx=9059, tau=0.99, symlog=False, out_dir=".", all_acts=all_acts_topq)
# save_activation_curve(12, 904, highlight_domain_idx=9059, tau=0.99, symlog=False, out_dir=".", all_acts=all_acts_topq)

Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L8_2677_sorted_curve.png


### main act curve

In [21]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def save_activation_curve(
    layer_i,
    latent_ind,
    tau=0.99,
    symlog=False,
    highlight_domain_idx=None,
    highlight_domain_idxs=None,
    highlight_color="orange",
    highlight_alpha=0.8,
    highlight_marker_size=18,
    highlight_edgecolor="black",
    highlight_edgewidth=0.4,
    sample_control=False,
    control_domain_range=(0, 10096),
    control_min_proteins=21,
    control_max_proteins=None,
    control_color="slategray",
    control_alpha=0.75,
    control_marker_size=20,
    control_edgecolor="black",
    control_edgewidth=0.35,
    control_seed=None,
    rng=None,
    domain_jitter=0.02,
    out_dir=None,
    all_acts=None,
):
    """Plot a sorted activation curve with optional domain overlays.

    highlight_domain_idx / highlight_domain_idxs can mark multiple columns;
    supply a list/tuple/ndarray for the latter. If highlight_color is a
    sequence, its entries are cycled across those domains.

    When sample_control=True, a control domain is drawn uniformly from
    control_domain_range (inclusive) that has ≥ control_min_proteins members
    and isn’t already highlighted. domain_jitter controls the vertical offset
    applied to each highlighted domain (fraction of the activation range).
    """
    flattened_id = to_flattened_id(int(layer_i), int(latent_ind))
    latent_i = subset_list.index(flattened_id)

    acts_source = all_acts if all_acts is not None else globals().get("all_acts")
    if acts_source is None:
        raise ValueError("Activation table `all_acts` must be provided or exist globally.")

    v = to_numpy(acts_source)[:, latent_i].astype(float)
    order = np.argsort(v)[::-1]
    s = v[order]
    N = s.size

    out_dir_use = globals().get("out_dir", Path("./")) if out_dir is None else Path(out_dir)
    out_dir_use.mkdir(parents=True, exist_ok=True)

    k_top = max(1, int(np.ceil((1 - float(tau)) * N)))

    if N > 1 and (s.max() - s.min()) > 0:
        x = np.linspace(0.0, 1.0, N)
        y = (s - s.min()) / (s.max() - s.min() + 1e-12)
        knee_idx = int(np.argmax(y - (1.0 - x)))
    else:
        knee_idx = 0

    highlight_ids = []
    if highlight_domain_idx is not None:
        highlight_ids.append(int(highlight_domain_idx))
    if highlight_domain_idxs is not None:
        if isinstance(highlight_domain_idxs, np.ndarray):
            highlight_ids.extend(int(i) for i in highlight_domain_idxs.flatten())
        elif isinstance(highlight_domain_idxs, (list, tuple, set)):
            highlight_ids.extend(int(i) for i in highlight_domain_idxs)
        else:
            highlight_ids.append(int(highlight_domain_idxs))
    seen = set()
    ordered_ids = []
    for idx in highlight_ids:
        if idx not in seen:
            ordered_ids.append(idx)
            seen.add(idx)

    domain_specs = [{"idx": idx, "source": "user"} for idx in ordered_ids]

    needs_properties = bool(domain_specs) or sample_control
    properties = None
    if needs_properties:
        properties = globals().get("all_properties")
        if properties is None:
            raise ValueError("Domain highlighting/control requires `all_properties`.")
        properties = to_numpy(properties)
        if properties.ndim != 2:
            raise ValueError("`all_properties` must be a 2-D array.")

    if sample_control and properties is not None:
        rng_use = rng if rng is not None else np.random.default_rng(control_seed)
        lower, upper = control_domain_range
        lower = max(int(lower), 0)
        upper = int(upper)
        if properties.shape[1] == 0:
            raise ValueError("`all_properties` has zero columns.")
        upper = min(upper, properties.shape[1] - 1)
        if lower > upper:
            print(f"No control sampled: range {control_domain_range} outside available columns.")
        else:
            candidates = []
            for col in range(lower, upper + 1):
                if col in seen:
                    continue
                n_present = int((properties[:, col] != 0).sum())
                if n_present < control_min_proteins:
                    continue
                if control_max_proteins is not None and n_present > control_max_proteins:
                    continue
                candidates.append((col, n_present))
            if not candidates:
                print(f"No eligible control domain in [{lower}, {upper}] with ≥{control_min_proteins} proteins.")
            else:
                pick = int((rng_use or np.random.default_rng()).integers(len(candidates)))
                control_idx, control_count = candidates[pick]
                domain_specs.append({"idx": control_idx, "source": "control", "count": control_count})
                seen.add(control_idx)
                print(f"Sampled control domain {control_idx} (n={control_count}).")

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(np.arange(1, N + 1), s, lw=1.2, color="#1f77b4")
    ax.axvline(k_top, color="crimson", ls="--", alpha=0.7, label=f"top {100*(1-float(tau)):.1f}%")
    ax.axvline(knee_idx + 1, color="teal", ls=":", alpha=0.7, label="knee")

    if domain_specs and properties is not None:
        color_cycle = list(highlight_color) if isinstance(highlight_color, (list, tuple, np.ndarray)) else [highlight_color]
        if not color_cycle:
            color_cycle = ["orange"]

        y_span = float(s.max() - s.min()) if np.isfinite(s.max() - s.min()) else 0.0
        if y_span == 0:
            y_span = max(float(np.abs(s).max()), 1.0)
        jitter_step = float(domain_jitter) * y_span if domain_jitter else 0.0

        for spec_idx, spec in enumerate(domain_specs):
            col = spec["idx"]
            source = spec.get("source", "user")

            if col < 0 or col >= properties.shape[1]:
                ax.scatter([], [], s=highlight_marker_size, color="gray", alpha=0.5, edgecolors="none",
                           label=f"domain {col} (out of range)")
                print(f"Domain index {col} out of range for all_properties (columns={properties.shape[1]}).")
                continue

            mask = properties[:, col] != 0
            mask_sorted = mask[order]
            idxs = np.flatnonzero(mask_sorted)

            n_total = int(mask.sum())
            n_top = int(mask_sorted[:k_top].sum())
            coverage = (n_top / n_total) if n_total else 0.0

            df_ann = globals().get("interpro_annotations_nonzero")
            label_name = None
            if df_ann is not None:
                try:
                    label_name = str(df_ann.iloc[col]["ENTRY_NAME"])
                except Exception:
                    pass

            if source == "control":
                point_color = control_color
                marker_size = control_marker_size
                alpha_val = control_alpha
                edge_color = control_edgecolor
                edge_width = control_edgewidth
                base = f"control {col}" if label_name is None else f"control {label_name} (idx {col})"
            else:
                point_color = color_cycle[spec_idx % len(color_cycle)]
                marker_size = highlight_marker_size
                alpha_val = highlight_alpha
                edge_color = highlight_edgecolor
                edge_width = highlight_edgewidth
                base = f"domain {col}" if label_name is None else f"{label_name} (idx {col})"

            label = f"{base}; n={n_total}; cov@tau={coverage:.0%}"
            offset = (spec_idx - (len(domain_specs) - 1) / 2.0) * jitter_step if jitter_step else 0.0
            scatter_args = dict(
                s=marker_size,
                color=point_color,
                alpha=alpha_val,
                edgecolors=edge_color,
                linewidths=edge_width,
                label=label,
                zorder=3,
            )

            if idxs.size:
                ax.scatter(idxs + 1, s[idxs] + offset, **scatter_args)
            else:
                ax.scatter([], [], **scatter_args)

    ax.set_xlabel("Proteins (sorted by latent activation, rank)")
    ax.set_ylabel("Activation")
    ax.set_xlim(1, N)
    if symlog:
        ax.set_yscale("symlog")
    ax.legend(frameon=False)
    ax.set_title(f"L{layer_i}-{latent_ind} (flat {flattened_id})")
    plt.tight_layout()

    out_path = out_dir_use / f"L{layer_i}_{latent_ind}_sorted_curve.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path.resolve()}")

def to_numpy(a):
    if torch is not None and isinstance(a, torch.Tensor):
        return a.detach().cpu().numpy()
    return np.asarray(a)


In [28]:
save_activation_curve(8, 2677, highlight_domain_idxs=[9278, 9059], tau=0.99, symlog=False, out_dir=".", all_acts=all_acts_topq, sample_control=True, highlight_color=["orange","mediumseagreen","royalblue"], domain_jitter=0.05, control_min_proteins=30, control_max_proteins=100, control_seed=12)

Sampled control domain 8743 (n=36).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L8_2677_sorted_curve.png


In [30]:
latent_specs = [
    (4, 798, 1206),
    (12, 2112, 9059),
    (8, 2677, 9278),
    (12, 3035, 8787),
    (8, 488, 9059),
    (12, 1256, 9005),
    (12, 3794, 9063),
    (12, 3536, 9063),
    (12, 2797, 8740),
    # (16, 1166),
    (8, 2775, 1284),
    (8, 2166, 9982),
    (12, 2472, 2373),
    # (12, 1082),
    # add more (layer_i, latent_ind) tuples here
]
for layer_i, latent_ind, domain_idx in latent_specs:
    save_activation_curve(layer_i, latent_ind, highlight_domain_idxs=[domain_idx, 9059], tau=0.99, symlog=False, out_dir=".", all_acts=all_acts_topq, sample_control=True, highlight_color=["orange","mediumseagreen"], domain_jitter=0.05, control_min_proteins=30, control_max_proteins=100, control_seed=12)

Sampled control domain 8743 (n=36).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L4_798_sorted_curve.png
Sampled control domain 8743 (n=36).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L12_2112_sorted_curve.png
Sampled control domain 8743 (n=36).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L8_2677_sorted_curve.png
Sampled control domain 8739 (n=77).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L12_3035_sorted_curve.png
Sampled control domain 8743 (n=36).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L8_488_sorted_curve.png
Sampled control domain 8743 (n=36).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L12_1256_sorted_curve.png
Sampled control domain 8743 (n=36).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/

In [40]:
save_activation_curve(12, 1377, highlight_domain_idxs=[9278, 9235, 9059], tau=0.99, symlog=False, out_dir=".", all_acts=all_acts_topq, sample_control=True, highlight_color=["orange","mediumseagreen", "royalblue"], domain_jitter=0.05, control_min_proteins=30, control_max_proteins=100, control_seed=12)

Sampled control domain 8739 (n=77).
Saved: /work/pi_jensen_umass_edu/jnainani_umass_edu/plm_circuits/notebooks/domain_corr/L12_1377_sorted_curve.png


### enrichment / odds ratio 

In [31]:
def to_numpy(a):
    if torch is not None and isinstance(a, torch.Tensor):
        return a.detach().cpu().numpy()
    return np.asarray(a)

def percentile_ranks(x: np.ndarray) -> np.ndarray:
    """Return percentile ranks in [0,1] with average ranks on ties."""
    # rankdata gives 1..N; convert to 0..1
    r = rankdata(x, method="average")
    return (r - 1) / (x.size - 1) if x.size > 1 else np.zeros_like(r, dtype=float)

def bonferroni(pvals: np.ndarray) -> np.ndarray:
    m = pvals.size
    out = pvals * m
    out[out > 1.0] = 1.0
    return out

def tail_enrichment_table(R: np.ndarray, in_mask: np.ndarray, tau: float = 0.99):
    """
    Build 2x2 counts for Fisher at top tail threshold tau on percentile R.
      a: in-domain & top-tail
      b: out-of-domain & top-tail
      c: in-domain & not top-tail
      d: out-of-domain & not top-tail
    """
    top_mask = R >= np.quantile(R, tau)  # top (1 - tau) fraction; robust to ties
    a = np.sum(in_mask & top_mask)
    b = np.sum(~in_mask & top_mask)
    c = np.sum(in_mask & ~top_mask)
    d = np.sum(~in_mask & ~top_mask)
    return a, b, c, d

def screen_domains_tail_enrichment(
    all_acts,
    all_properties,
    latent_i: int,
    domain_indices: np.ndarray | list | None = None,
    tau: float = 0.99,
    n_in_min: int = 20,
    n_out_min: int = 200,
    use_bonferroni: bool = True,
    return_tau_all_in: bool = True,  # NEW
    coverage_fracs: tuple[float, ...] | None = (0.95, 0.90),  # NEW
):
    X = to_numpy(all_acts)
    D = to_numpy(all_properties)

    assert X.ndim == 2, "all_acts must be 2D [N, L]"
    assert D.ndim == 2, "all_properties must be 2D [N, M]"
    N = X.shape[0]

    # 1) Percentile ranks (scale-agnostic) for the chosen latent
    R = percentile_ranks(X[:, latent_i])

    # Domain set to consider
    if domain_indices is None:
        domain_indices = np.arange(D.shape[1])
    else:
        domain_indices = np.asarray(domain_indices, dtype=int)

    # Binarize domain membership (nonzero -> True)
    D_bin = D[:, domain_indices] != 0

    # Precompute the top-mask once (so Fisher thresholds are consistent across domains)
    # This uses same tau for every domain.
    thr = np.quantile(R, tau)
    top_mask = R >= thr
    not_top_mask = ~top_mask

    # Precompute totals
    top_count = int(top_mask.sum())
    not_top_count = N - top_count

    # Counts per domain (vectorized where possible)
    in_counts = D_bin.sum(axis=0)                    # n_in per domain
    out_counts = N - in_counts                       # n_out per domain

    # Filter domains by minimum sizes
    ok = (in_counts >= n_in_min) & (out_counts >= n_out_min)
    idxs = domain_indices[ok]
    if idxs.size == 0:
        cols = [
            "domain_idx", "n_in", "n_out",
            "a_top_in", "b_top_out", "c_not_top_in", "d_not_top_out",
            "odds_ratio", "fisher_p", "p_adj",
            "median_pos_percentile"
        ]
        if return_tau_all_in:
            cols.append("tau_all_in")
        return pd.DataFrame(columns=cols)

    D_ok = D_bin[:, ok]  # [N, K]
    n_in = in_counts[ok]
    n_out = out_counts[ok]

    # Vectorized a, c via matrix multiplications / sums
    a_vec = (top_mask[:, None] & D_ok).sum(axis=0)        # top & in
    c_vec = (not_top_mask[:, None] & D_ok).sum(axis=0)    # not top & in
    b_vec = top_count - a_vec                              # top & out
    d_vec = not_top_count - c_vec                          # not top & out

    # Fisher p-values and OR (loop per domain; fast enough for ~10k)
    pvals = np.empty(a_vec.size, dtype=float)
    ORs = np.empty(a_vec.size, dtype=float)
    for k in range(a_vec.size):
        a, b, c, d = int(a_vec[k]), int(b_vec[k]), int(c_vec[k]), int(d_vec[k])
        # Add a tiny continuity guard if any cell is zero when computing OR
        if a == 0 or b == 0 or c == 0 or d == 0:
            OR = ((a + 0.5) / (c + 0.5)) / ((b + 0.5) / (d + 0.5))
        else:
            OR = (a / c) / (b / d)
        ORs[k] = OR
        _, p = fisher_exact([[a, b], [c, d]], alternative="greater")
        pvals[k] = p

    p_adj = bonferroni(pvals) if use_bonferroni else pvals  # plug-in BH here if you later switch

    # Median positive percentile per domain
    med_pos = np.empty(a_vec.size, dtype=float)
    for k in range(a_vec.size):
        mask_k = D_ok[:, k]
        r_in = R[mask_k]
        med_pos[k] = float(np.median(r_in)) if r_in.size > 0 else np.nan

    data = {
        "domain_idx": idxs,
        "domain_name": [interpro_annotations_nonzero.iloc[idx]["ENTRY_NAME"] for idx in idxs],
        "n_in": n_in.astype(int),
        "n_out": n_out.astype(int),
        "a_top_in": a_vec.astype(int),
        "b_top_out": b_vec.astype(int),
        "c_not_top_in": c_vec.astype(int),
        "d_not_top_out": d_vec.astype(int),
        "odds_ratio": ORs,
        "fisher_p": pvals,
        "p_adj": p_adj,
        "median_pos_percentile": med_pos,
    }
        # NEW: tau at which ≥p of in-domain are in the top tail (p in coverage_fracs)
    if coverage_fracs:
        coverage_fracs = tuple(sorted(coverage_fracs, reverse=True))  # e.g., (0.95, 0.90)
        sorted_R = np.sort(R)
        denom = max(N - 1, 1)

        # Collect per-domain lower-tail quantiles r_q for q = 1 - p
        r_q = {p: np.empty(a_vec.size, dtype=float) for p in coverage_fracs}

        for k in range(a_vec.size):
            mask_k = D_ok[:, k]
            r_in = R[mask_k]
            if r_in.size == 0:
                for p in coverage_fracs:
                    r_q[p][k] = np.nan
            else:
                for p in coverage_fracs:
                    q = 1.0 - p  # need thr <= q-quantile to cover ≥p of in-domain
                    r_q[p][k] = float(np.quantile(r_in, q))

        for p in coverage_fracs:
            idx = np.searchsorted(sorted_R, r_q[p], side="right") - 1
            tau_cover = np.clip(idx / denom, 0.0, 1.0)
            data[f"tau_cover_{int(p*100)}"] = tau_cover

    # NEW: per-domain largest tau such that c == 0 (all in-domain are in top tail)
    if return_tau_all_in:
        sorted_R = np.sort(R)  # ascending
        # per-domain minimum in-domain percentile
        r_min_in = np.where(D_ok, R[:, None], np.inf).min(axis=0)
        # last index in sorted_R where value <= r_min_in
        last_le_idx = np.searchsorted(sorted_R, r_min_in, side="right") - 1
        # convert to tau using numpy's default quantile definition (linear over indices 0..N-1)
        denom = max(N - 1, 1)
        tau_all_in = np.clip(last_le_idx / denom, 0.0, 1.0)
        data["tau_all_in"] = tau_all_in

    df = pd.DataFrame(data)

    # Suggested default ranking: strongest practical signals first
    df = df.sort_values(
        by=["p_adj", "odds_ratio", "median_pos_percentile"],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    return df

In [ ]:
# layer_i = 8
# latent_ind = 2677
# flattened_id = to_flattened_id(int(layer_i), int(latent_ind))
# latent_i = subset_list.index(flattened_id)
# tail_df = screen_domains_tail_enrichment(all_acts_topq, all_properties, latent_i, tau=0.98)
# hits = tail_df[(tail_df["p_adj"] <= 0.05) & (tail_df["odds_ratio"] >= 5)]
# hits.head(25)

,domain_idx,domain_name,n_in,n_out,a_top_in,b_top_out,c_not_top_in,d_not_top_out,odds_ratio,fisher_p,p_adj,median_pos_percentile,tau_cover_95,tau_cover_90,tau_all_in
0,9278,NAD(P)-binding domain superfamily,206,9794,91,109,115,9685,70.309932,3.290869e-106,9.346068e-104,0.977548,0.942794,0.946095,0.509751
1,9235,FAD/NAD(P)-binding domain superfamily,75,9925,55,145,20,9780,185.482759,6.842719e-80,1.943332e-77,0.989499,0.963396,0.969897,0.813181
2,4311,Short-chain dehydrogenase/reductase SDR,26,9974,26,174,0,9800,2976.656160,1.267983e-45,3.601071e-43,0.994999,0.989699,0.990999,0.987699
3,9059,Alpha/Beta hydrolase fold,63,9937,33,167,30,9770,64.353293,2.908813e-40,8.261029e-38,0.980998,0.942794,0.952895,0.935794
4,10002,"Aminoacid dehydrogenase-like, N-terminal domai...",30,9970,13,187,17,9783,40.005977,4.926904e-15,1.399241e-12,0.964096,0.934793,0.941994,0.922692
5,8558,"6-phosphogluconate dehydrogenase-like, C-termi...",31,9969,13,187,18,9782,37.779560,8.337689e-15,2.367904e-12,0.979348,0.780178,0.940694,0.509751
6,9063,S-adenosyl-L-methionine-dependent methyltransf...,179,9821,19,181,160,9640,6.324586,2.634619e-09,7.482319e-07,0.960996,0.909191,0.930793,0.661266


### mwu 

In [32]:
def _to_numpy(x):
    try:
        import torch
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(x)

def do_mwu_test_df(all_acts, layer_ind, latent_ind, domain_indices=None, subset_list=subset_list,
                   sort_by="p", ascending=True, alternative="greater"):
    flattened_id = to_flattened_id(int(layer_ind), int(latent_ind))
    latent_i = subset_list.index(flattened_id)

    # Ensure arrays are NumPy
    all_acts_np = _to_numpy(all_acts)

    # Default to all domains based on all_properties
    if domain_indices is None:
        domain_indices = np.arange(all_properties.shape[1])
    else:
        domain_indices = np.asarray(domain_indices, dtype=int)

    rows = []
    for domain_idx in tqdm(domain_indices):
        x_mask = (all_properties[:, domain_idx] != 0)
        y_mask = ~x_mask

        x_raw = np.asarray(all_acts_np[x_mask, latent_i], dtype=float).ravel()
        y_raw = np.asarray(all_acts_np[y_mask, latent_i], dtype=float).ravel()

        if x_raw.size == 0 or y_raw.size == 0:
            p = np.nan
            mean_pos = np.nan
            mean_neg = np.nan
            n_pos = int(x_raw.size)
            n_neg = int(y_raw.size)
        else:
            _, p = mannwhitneyu(x_raw, y_raw, alternative=alternative)
            mean_pos = float(np.mean(x_raw))
            mean_neg = float(np.mean(y_raw))
            n_pos = int(x_raw.size)
            n_neg = int(y_raw.size)

        try:
            entry_name = interpro_annotations_nonzero.iloc[int(domain_idx)]["ENTRY_NAME"]
        except Exception:
            entry_name = None

        rows.append({
            "layer_ind": int(layer_ind),
            "latent_ind": int(latent_ind),
            "flattened_id": int(flattened_id),
            "latent_i": int(latent_i),
            "domain_idx": int(domain_idx),
            "entry_name": entry_name,
            "p": p,
            "mean_pos": mean_pos,
            "mean_neg": mean_neg,
            "delta": (mean_pos - mean_neg) if np.isfinite(mean_pos) and np.isfinite(mean_neg) else np.nan,
            "n_pos": n_pos,
            "n_neg": n_neg,
        })

    df = pd.DataFrame(rows)
    if sort_by is not None and sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=ascending).reset_index(drop=True)
    return df

In [9]:
# Example usage
# mwu_df = do_mwu_test_df(all_acts_topq, 8, 2677)
# mwu_df.head(25)

## manual latents of interest 

In [33]:
latent_specs = [
    (4, 798),
    (12, 2112),
    (8, 2677),
    (12, 3035),
    (8, 488),
    (12, 1256),
    (12, 3794),
    (12, 3536),
    (12, 2797),
    (16, 1166),
    (8, 2775),
    (8, 2166),
    (12, 2472),
    (12, 1082),
    # add more (layer_i, latent_ind) tuples here
]

relevant_interpro_entries = [
    "IPR036291",
    "IPR029063", 
    "IPR036188",
    "IPR029058",
    "IPR002347",
    "IPR046346",
    "IPR023753",
    "IPR020904",
    "IPR006151",
    "IPR004101",
    "IPR013221",
    "IPR000073",
    "IPR040131",
    "IPR002218",
    "IPR015955",
    "IPR001236",
    "IPR022383",
    "IPR001557",
    "IPR020630",
    "IPR020631",
    "IPR000672"
]

# Attempt 1 at summary

In [34]:
import numpy as np
import pandas as pd
from scipy.stats import rankdata, mannwhitneyu, fisher_exact

# ---------- small utilities ----------
def to_numpy(a):
    try:
        import torch
        if isinstance(a, torch.Tensor):
            return a.detach().cpu().numpy()
    except Exception:
        pass
    return np.asarray(a)

def percentile_ranks(x: np.ndarray) -> np.ndarray:
    r = rankdata(x, method="average")
    return (r - 1) / (x.size - 1) if x.size > 1 else np.zeros_like(r, dtype=float)

def bonferroni_local(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    m = pvals.size
    out = pvals * m
    out[out > 1.0] = 1.0
    return out

# ---------- top-k metrics (precision, lift, Fisher/OR) ----------
def topk_metrics(scores: np.ndarray, in_mask: np.ndarray, k: int):
    """Return precision@k, lift@k, (a,b,c,d), OR (Haldane), Fisher p (one-sided)."""
    scores = to_numpy(scores).astype(float).ravel()
    in_mask = to_numpy(in_mask).astype(bool).ravel()
    N = scores.size
    assert in_mask.size == N and 1 <= k <= N

    order = np.argsort(scores)[::-1]
    top = order[:k]
    rest = order[k:]

    TP = int(in_mask[top].sum())
    FP = k - TP
    FN = int(in_mask[rest].sum())
    TN = N - k - FN

    p = TP / k if k > 0 else np.nan
    prev = in_mask.mean() if N > 0 else np.nan
    lift = (p / prev) if prev > 0 else np.inf

    # Odds ratio with Haldane–Anscombe correction if any zero
    a,b,c,d = TP, FP, FN, TN
    if 0 in (a,b,c,d):
        OR = ((a + 0.5) / (c + 0.5)) / ((b + 0.5) / (d + 0.5))
    else:
        OR = (a / c) / (b / d)

    _, fisher_p = fisher_exact([[a, b], [c, d]], alternative="greater")
    return {
        "k": k, "precision_k": p, "lift_k": lift,
        "a_top_in": a, "b_top_out": b, "c_not_top_in": c, "d_not_top_out": d,
        "or_topk": OR, "fisher_p_topk": fisher_p
    }

# ---------- MWU (plus optional AUC for interpretability) ----------
def mwu_test(scores_in: np.ndarray, scores_out: np.ndarray, alternative="greater"):
    if scores_in.size == 0 or scores_out.size == 0:
        return np.nan, np.nan
    U, p = mannwhitneyu(scores_in, scores_out, alternative=alternative)
    auc = U / (scores_in.size * scores_out.size)
    return p, auc

# ---------- consolidate per-latent analysis (P1 + P2) ----------
def analyze_latent_per_domain(
    all_acts, all_properties, latent_i: int,
    entry_names=None,
    k_multipliers=(1, 2),       # use k = mult * n_in
    use_bonferroni=True,
    min_in=20, min_out=200,
    summarized="topq"           # just for bookkeeping in the output
):
    """
    Returns a single DataFrame for this latent with:
    - n_in/n_out
    - MWU p and Bonf (P1)
    - precision/lift at k=n_in and 2*n_in (P2)
    - Fisher p + OR on the same 2x2 at each k (P2)
    """
    X = to_numpy(all_acts)
    D = (to_numpy(all_properties) != 0)
    N, M = D.shape

    scores = X[:, latent_i].astype(float)
    prevs = D.sum(axis=0) / N
    n_in = D.sum(axis=0)
    n_out = N - n_in
    ok = (n_in >= min_in) & (n_out >= min_out)

    idxs = np.where(ok)[0]
    if idxs.size == 0:
        cols = ["domain_idx","entry_name","n_in","n_out","mwu_p","mwu_p_adj","mwu_auc"]
        for mult in k_multipliers:
            cols += [f"k_{mult}x","precision","lift","fisher_p","fisher_p_adj","or_topk"]
        return pd.DataFrame(columns=cols)

    # --- P1: MWU per domain (correct within this latent's tested domains) ---
    mwu_p = np.empty(idxs.size, dtype=float)
    mwu_auc = np.empty(idxs.size, dtype=float)
    for j, dom in enumerate(idxs):
        mask = D[:, dom]
        p, auc = mwu_test(scores[mask], scores[~mask], alternative="greater")
        mwu_p[j] = p
        mwu_auc[j] = auc

    mwu_p_adj = bonferroni_local(mwu_p) if use_bonferroni else mwu_p

    # --- P2: top-k metrics at k = n_in and 2*n_in (no knee) ---
    # We also Bonferroni-correct Fisher p's within this same set.
    results = []
    fisher_cols = []
    for j, dom in enumerate(idxs):
        row = {
            "domain_idx": int(dom),
            "entry_name": (entry_names[dom] if entry_names is not None else None),
            "n_in": int(n_in[dom]),
            "n_out": int(n_out[dom]),
            "mwu_p": mwu_p[j],
            "mwu_p_adj": mwu_p_adj[j],
            "mwu_auc": mwu_auc[j],
            "prevalence": float(prevs[dom]),
            "summary_kind": summarized,
        }
        # compute once and store; we'll collect fisher p's to adjust jointly
        fisher_ps_this = []
        for mult in k_multipliers:
            k = int(max(1, min(scores.size, mult * n_in[dom])))
            tm = topk_metrics(scores, D[:, dom], k)
            row.update({
                f"k_{mult}x": k,
                f"precision_{mult}x": tm["precision_k"],
                f"lift_{mult}x": tm["lift_k"],
                f"or_topk_{mult}x": tm["or_topk"],
                f"fisher_p_{mult}x": tm["fisher_p_topk"],
            })
            fisher_ps_this.append(tm["fisher_p_topk"])
            fisher_cols.append((len(results), mult))  # remember where to write adj p
        results.append(row)

    df = pd.DataFrame(results)

    # Bonferroni for Fisher (within the same per-latent family)
    # fisher_all = df[[f"fisher_p_{mult}x" for mult in k_multipliers]].to_numpy().ravel()
    # print("Bonferroni m for Fisher:", np.asarray(fisher_all, dtype=float).size, "min p:", fisher_all.min())
    # fisher_all_adj = bonferroni_local(fisher_all) if use_bonferroni else fisher_all
    # # write back
    # ptr = 0
    # for mult in k_multipliers:
    #     adj_vals = fisher_all_adj[ptr:ptr+df.shape[0]]
    #     df[f"fisher_p_adj_{mult}x"] = adj_vals
    #     ptr += df.shape[0]
    if use_bonferroni:
        for mult in k_multipliers:
            pcol = f"fisher_p_{mult}x"
            df[f"fisher_p_adj_{mult}x"] = bonferroni_local(df[pcol].to_numpy())
    else:
        for mult in k_multipliers:
            pcol = f"fisher_p_{mult}x"
            df[f"fisher_p_adj_{mult}x"] = df[pcol].to_numpy()

    # convenience sort: by mwu_p_adj then precision at 1x
    if f"precision_{k_multipliers[0]}x" in df.columns:
        df = df.sort_values(
            by=["mwu_p_adj", f"precision_{k_multipliers[0]}x", f"lift_{k_multipliers[0]}x"],
            ascending=[True, False, False]
        ).reset_index(drop=True)

    return df


In [46]:
layer_i = 8
latent_ind = 488 
flattened_id = to_flattened_id(int(layer_i), int(latent_ind))
latent_i = subset_list.index(flattened_id)

entry_names = list(interpro_annotations_nonzero["ENTRY_NAME"])
df_latent = analyze_latent_per_domain(
    all_acts=all_acts_topq,        # e.g., your top-q summary
    all_properties=all_properties, # [N, M] binary/proportional
    latent_i=latent_i,
    entry_names=entry_names,
    k_multipliers=(1, 2),          # k = n_in and 2*n_in
    use_bonferroni=True,
    min_in=20, min_out=200,
    summarized="topq"
)

# Gate for "domain detector" candidates (no knee, Bonferroni-simple):
candidates = df_latent[
    (df_latent["mwu_p_adj"] <= 0.05) &
    (
      ((df_latent["precision_1x"] >= 0.5) & (df_latent["lift_1x"] >= 3)) |
      ((df_latent["precision_2x"] >= 0.5) & (df_latent["lift_2x"] >= 3))
    ) &
    (
      (df_latent["fisher_p_adj_1x"] <= 0.05) | (df_latent["fisher_p_adj_2x"] <= 0.05)
    )
]
df_latent.head(25)

,domain_idx,entry_name,n_in,n_out,mwu_p,mwu_p_adj,mwu_auc,prevalence,summary_kind,k_1x,...,lift_1x,or_topk_1x,fisher_p_1x,k_2x,precision_2x,lift_2x,or_topk_2x,fisher_p_2x,fisher_p_adj_1x,fisher_p_adj_2x
0,9059,Alpha/Beta hydrolase fold,63,9937,0.000000e+00,0.000000e+00,0.971049,0.0063,topq,63,...,100.781053,749.640832,7.399567e-79,126,0.373016,59.208869,366.555380,9.554849e-80,2.101477e-76,2.713577e-77
1,1395,Small GTP-binding domain,128,9872,1.022301e-81,2.903336e-79,0.656264,0.0128,topq,128,...,10.375977,13.467819,4.207919e-13,256,0.125000,9.765625,14.357143,2.848967e-23,1.195049e-10,8.091068e-21
2,8588,Translation elongation factor EF1A/initiation ...,23,9977,3.252699e-65,9.237665e-63,0.826451,0.0023,topq,23,...,0.000000,9.012675,1.000000e+00,46,0.000000,0.000000,4.544269,1.000000e+00,1.000000e+00,1.000000e+00
3,1458,GTP binding domain,62,9938,2.182739e-31,6.198979e-29,0.635772,0.0062,topq,62,...,2.601457,2.654394,3.207603e-01,124,0.024194,3.902185,4.125368,4.146482e-02,1.000000e+00,1.000000e+00
4,1301,"Translation elongation factor EFTu-like, domain 2",55,9945,6.004822e-28,1.705369e-25,0.635418,0.0055,topq,55,...,0.000000,1.605470,1.000000e+00,110,0.000000,0.000000,0.801883,1.000000e+00,1.000000e+00,1.000000e+00
5,609,"Tr-type G domain, conserved site",56,9944,2.053608e-27,5.832248e-25,0.632825,0.0056,topq,56,...,0.000000,1.548829,1.000000e+00,112,0.000000,0.000000,0.773451,1.000000e+00,1.000000e+00,1.000000e+00
6,9005,P-loop containing nucleoside triphosphate hydr...,652,9348,3.570832e-27,1.014116e-24,0.539959,0.0652,topq,652,...,2.140653,2.540717,1.246937e-12,1304,0.092791,1.423181,1.572759,2.244843e-05,3.541302e-10,6.375355e-03
7,782,Translational (tr)-type GTP-binding domain,70,9930,1.105017e-23,3.138248e-21,0.609839,0.0070,topq,70,...,0.000000,0.991952,1.000000e+00,140,0.000000,0.000000,0.494208,1.000000e+00,1.000000e+00,1.000000e+00
8,10002,"Aminoacid dehydrogenase-like, N-terminal domai...",30,9970,2.360241e-19,6.703085e-17,0.649891,0.0030,topq,30,...,0.000000,5.342919,1.000000e+00,60,0.016667,5.555556,5.792519,1.654021e-01,1.000000e+00,1.000000e+00
9,9278,NAD(P)-binding domain superfamily,206,9794,3.456003e-13,9.815049e-11,0.546469,0.0206,topq,206,...,3.770384,4.256620,5.185122e-06,412,0.065534,3.181261,3.686324,8.781981e-08,1.472575e-03,2.494083e-05


In [47]:
candidates.head(25)

,domain_idx,entry_name,n_in,n_out,mwu_p,mwu_p_adj,mwu_auc,prevalence,summary_kind,k_1x,...,lift_1x,or_topk_1x,fisher_p_1x,k_2x,precision_2x,lift_2x,or_topk_2x,fisher_p_2x,fisher_p_adj_1x,fisher_p_adj_2x
0,9059,Alpha/Beta hydrolase fold,63,9937,0.0,0.0,0.971049,0.0063,topq,63,...,100.781053,749.640832,7.399567e-79,126,0.373016,59.208869,366.55538,9.554849e-80,2.101477e-76,2.713577e-77


In [32]:
df_latent.head(25)

,domain_idx,entry_name,n_in,n_out,mwu_p,mwu_p_adj,mwu_auc,prevalence,summary_kind,k_1x,...,lift_1x,or_topk_1x,fisher_p_1x,k_2x,precision_2x,lift_2x,or_topk_2x,fisher_p_2x,fisher_p_adj_1x,fisher_p_adj_2x
0,9278,NAD(P)-binding domain superfamily,206,9794,1.299686e-123,3.691108e-121,0.979828,0.0206,topq,206,...,22.151004,72.553253,3.230261e-110,412,0.390777,18.969743,136.026826,1.135466e-193,2.245639e-21,1.0
1,9063,S-adenosyl-L-methionine-dependent methyltransf...,179,9821,7.292740e-101,2.071138e-98,0.963358,0.0179,topq,179,...,4.057302,4.554906,1.879559e-05,358,0.215084,12.015855,25.629056,3.834208e-65,1.000000e+00,1.0
2,9235,FAD/NAD(P)-binding domain superfamily,75,9925,1.393282e-48,3.956921e-46,0.988376,0.0075,topq,75,...,49.777778,125.207786,2.528614e-42,150,0.333333,44.444444,196.500000,2.840775e-76,1.000000e+00,1.0
3,9005,P-loop containing nucleoside triphosphate hydr...,652,9348,7.761982e-45,2.204403e-42,0.663657,0.0652,topq,652,...,0.164666,0.146436,1.000000e+00,1304,0.091258,1.399657,1.537982,5.658429e-05,1.000000e+00,1.0
4,9059,Alpha/Beta hydrolase fold,63,9937,6.868473e-40,1.950646e-37,0.980224,0.0063,topq,63,...,22.675737,30.503086,1.567319e-10,126,0.190476,30.234316,59.336350,2.702928e-30,1.000000e+00,1.0
5,9945,"ATPase, nucleotide binding domain",95,9905,2.118368e-36,6.016164e-34,0.873217,0.0095,topq,95,...,0.000000,0.537842,1.000000e+00,190,0.000000,0.000000,0.267016,1.000000e+00,1.000000e+00,1.0
6,9062,Class I glutamine amidotransferase-like,45,9955,1.943480e-21,5.519482e-19,0.906842,0.0045,topq,45,...,0.000000,2.393552,1.000000e+00,90,0.000000,0.000000,1.197924,1.000000e+00,1.000000e+00,1.0
7,9541,Ribonuclease Z/Hydroxyacylglutathione hydrolas...,35,9965,2.613455e-20,7.422214e-18,0.947578,0.0035,topq,35,...,0.000000,3.939893,1.000000e+00,70,0.000000,0.000000,1.976925,1.000000e+00,1.000000e+00,1.0
8,887,Metallo-beta-lactamase,34,9966,8.160440e-20,2.317565e-17,0.947957,0.0034,topq,34,...,0.000000,4.172443,1.000000e+00,68,0.000000,0.000000,2.094256,1.000000e+00,1.000000e+00,1.0
9,9050,Nucleotide-diphospho-sugar transferases,50,9950,1.293321e-19,3.673031e-17,0.867625,0.0050,topq,50,...,0.000000,1.941084,1.000000e+00,100,0.000000,0.000000,0.970445,1.000000e+00,1.000000e+00,1.0


In [37]:
df_latent.head(100).to_csv("latent_domain_correlations_l8_488.csv", index=False)

# iterate

In [48]:
# --- CONFIG you can tweak ---
K_MULTS = (1, 2)                 # evaluate at k = n_in and 2*n_in
MIN_IN, MIN_OUT = 20, 200
USE_BONFERRONI = True
SUMMARY_KIND = "topq"            # just a label in the output
ALL_LATENTS = list(range(len(subset_list)))  # iterate over your ~383 latents
OUT_ALL = "latent_domain_metrics_all.csv"
OUT_CANDS = "latent_domain_candidates.csv"

# Optional: domain names
entry_names = list(interpro_annotations_nonzero["ENTRY_NAME"])

# --- main loop ---
all_rows = []
errors = []

for latent_i in tqdm(ALL_LATENTS, desc="Analyzing latents"):
    try:
        layer_id, latent_ind = get_layer_and_latent(latent_i, subset_list)
        flattened_id = int(subset_list[latent_i])

        df_latent = analyze_latent_per_domain(
            all_acts=all_acts_topq,              # summarized activations (e.g., top-q)
            all_properties=all_properties,       # [N, M] domain indicators
            latent_i=latent_i,
            entry_names=entry_names,
            k_multipliers=K_MULTS,
            use_bonferroni=USE_BONFERRONI,
            min_in=MIN_IN, min_out=MIN_OUT,
            summarized=SUMMARY_KIND
        )

        if df_latent.empty:
            continue

        # annotate with latent identifiers
        df_latent.insert(0, "layer_id", layer_id)
        df_latent.insert(1, "latent_ind", latent_ind)
        df_latent.insert(2, "flattened_id", flattened_id)
        df_latent.insert(3, "latent_i", latent_i)

        all_rows.append(df_latent)

    except Exception as e:
        errors.append((latent_i, str(e)))

# --- concatenate & save ---
if len(all_rows):
    df_all = pd.concat(all_rows, axis=0, ignore_index=True)

    # Consistent sort: strongest global separation then best top-k utility
    base_sort = ["mwu_p_adj"]
    if f"precision_{K_MULTS[0]}x" in df_all.columns:
        base_sort += [f"precision_{K_MULTS[0]}x", f"lift_{K_MULTS[0]}x"]
    df_all = df_all.sort_values(by=base_sort, ascending=[True, False, False], kind="mergesort")

    # df_all.to_csv(OUT_ALL, index=False)
    print(f"Saved ALL metrics to: {OUT_ALL}  (rows={len(df_all)})")
else:
    df_all = pd.DataFrame()
    print("No rows produced — check inputs and filters (min_in/min_out).")

# --- candidate gate (your rule) ---
if not df_all.empty:
    gate = (
        (df_all["mwu_p_adj"] <= 0.05) &
        (
            ((df_all.get("precision_1x", 0) >= 0.5) & (df_all.get("lift_1x", 0) >= 3)) |
            ((df_all.get("precision_2x", 0) >= 0.5) & (df_all.get("lift_2x", 0) >= 3))
        ) &
        (
            (df_all.get("fisher_p_adj_1x", 1.0) <= 0.05) |
            (df_all.get("fisher_p_adj_2x", 1.0) <= 0.05)
        )
    )
    df_cands = df_all.loc[gate].copy()
    df_cands.to_csv(OUT_CANDS, index=False)
    print(f"Saved CANDIDATES to: {OUT_CANDS}  (rows={len(df_cands)})")

# --- any failures? ---
if errors:
    print("Errors encountered on latents:")
    for li, msg in errors[:10]:
        print(f"  latent_i={li}: {msg}")
    if len(errors) > 10:
        print(f"  ... and {len(errors) - 10} more")

Analyzing latents: 100%|██████████| 383/383 [25:07<00:00,  3.94s/it]

Saved ALL metrics to: latent_domain_metrics_all.csv  (rows=108772)
Saved CANDIDATES to: latent_domain_candidates.csv  (rows=192)


# domain overlap

In [35]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

def domain_overlap_report(
    all_properties: np.ndarray,
    domain_indices: list[int],
    entry_names: list[str] | None = None,
    protein_ids: list[str] | None = None,   # optional: to list overlapping proteins by ID
):
    """
    Build overlap summaries for a given list of InterPro domain indices.
    Returns:
      summary_df    : per-domain counts & prevalence
      pairwise_df   : long-form pairwise overlaps (A,B)
      multiway_info : dict with union/intersection masks & sizes
    """
    D = (np.asarray(all_properties) != 0)
    N, M = D.shape

    idx = np.array(domain_indices, dtype=int)
    if np.any((idx < 0) | (idx >= M)):
        bad = idx[(idx < 0) | (idx >= M)]
        raise IndexError(f"Invalid domain indices: {bad.tolist()} (M={M})")

    # Per-domain masks
    masks = {i: D[:, i] for i in idx}
    names = {i: (entry_names[i] if entry_names is not None else str(i)) for i in idx}

    # Per-domain summary
    n_in = {i: int(masks[i].sum()) for i in idx}
    prev = {i: n_in[i] / N for i in idx}
    summary_df = pd.DataFrame({
        "domain_idx": idx,
        "entry_name": [names[i] for i in idx],
        "n_in": [n_in[i] for i in idx],
        "prevalence": [prev[i] for i in idx],
    }).sort_values("prevalence", ascending=False, kind="mergesort").reset_index(drop=True)

    # Pairwise overlaps
    rows = []
    for i_pos, i in enumerate(idx):
        Ai = masks[i]
        ni = n_in[i]
        for j in idx[i_pos+1:]:
            Aj = masks[j]
            nj = n_in[j]

            inter = int(np.logical_and(Ai, Aj).sum())
            union = int(np.logical_or(Ai, Aj).sum())
            only_i = ni - inter
            only_j = nj - inter
            neither = N - union

            jacc = inter / union if union > 0 else np.nan
            pA_given_B = inter / nj if nj > 0 else np.nan
            pB_given_A = inter / ni if ni > 0 else np.nan

            # Fisher exact on 2×2 table: [[Ai∩Aj, Ai∩¬Aj],[¬Ai∩Aj, ¬Ai∩¬Aj]]
            table = [[inter, only_i],[only_j, neither]]
            _, fisher_p = fisher_exact(table, alternative="greater")

            rows.append({
                "A_idx": i, "A_name": names[i], "A_n": ni,
                "B_idx": j, "B_name": names[j], "B_n": nj,
                "intersect": inter, "union": union,
                "only_A": only_i, "only_B": only_j, "neither": neither,
                "jaccard": jacc,
                "P(A|B)": pA_given_B, "P(B|A)": pB_given_A,
                "fisher_p": fisher_p,
            })
    pairwise_df = pd.DataFrame(rows).sort_values(
        ["jaccard", "intersect"], ascending=[False, False], kind="mergesort"
    ).reset_index(drop=True)

    # Multiway intersection/union across ALL provided domains
    if len(idx) > 0:
        union_mask = np.logical_or.reduce([masks[i] for i in idx])
        inter_mask = np.logical_and.reduce([masks[i] for i in idx])
    else:
        union_mask = np.zeros(N, dtype=bool)
        inter_mask = np.zeros(N, dtype=bool)

    multiway_info = {
        "N": int(N),
        "k_domains": int(len(idx)),
        "union_size": int(union_mask.sum()),
        "intersection_size": int(inter_mask.sum()),
        "union_mask": union_mask,         # reuse for downstream filtering
        "intersection_mask": inter_mask,  # idem
    }

    # Optional: list the protein IDs that are in the intersection
    if protein_ids is not None:
        ids = np.asarray(protein_ids)
        multiway_info["intersection_protein_ids"] = ids[inter_mask].tolist()
        multiway_info["union_protein_ids"] = ids[union_mask].tolist()

    return summary_df, pairwise_df, multiway_info


In [41]:
domain_list = [9278, 9235] #[8787,10047,9289, 448, 10082]  # your example
entry_names = list(interpro_annotations_nonzero["ENTRY_NAME"])  # if available

summary_df, pairwise_df, multiway = domain_overlap_report(
    all_properties=all_properties, 
    domain_indices=domain_list, 
    entry_names=entry_names,
    protein_ids=None  # or a list of UniProt IDs aligned to rows of all_properties
)

pairwise_df.head(25)


,A_idx,A_name,A_n,B_idx,B_name,B_n,intersect,union,only_A,only_B,neither,jaccard,P(A|B),P(B|A),fisher_p
0,9278,NAD(P)-binding domain superfamily,206,9235,FAD/NAD(P)-binding domain superfamily,75,0,281,206,75,9719,0.0,0.0,0.0,1.0


In [52]:
pairwise_df.head()

,A_idx,A_name,A_n,B_idx,B_name,B_n,intersect,union,only_A,only_B,neither,jaccard,P(A|B),P(B|A),fisher_p
0,1395,Small GTP-binding domain,128,1458,GTP binding domain,62,41,149,87,21,9851,0.275168,0.66129,0.320312,3.054509e-65
1,9005,P-loop containing nucleoside triphosphate hydr...,652,1395,Small GTP-binding domain,128,128,652,524,0,9348,0.196319,1.00000,0.196319,5.916888e-158
2,9005,P-loop containing nucleoside triphosphate hydr...,652,1458,GTP binding domain,62,62,652,590,0,9348,0.095092,1.00000,0.095092,1.838542e-75


# Span overlap analysis

1. sample proteins from a domain 
2. gather spans from interpro 
3. check for odds ratio of activations wrt spans

## helpers

In [36]:


import requests
import pandas as pd
def fetch_uniprot_entry(acc: str) -> dict:
    url = f"https://rest.uniprot.org/uniprotkb/{acc}"
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=20)
    r.raise_for_status()
    return r.json()

def fetch_uniprot_fasta(acc: str) -> str:
    url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
    r = requests.get(url, timeout=20)
    r.raise_for_status()
    return r.text

def get_sae(layer: int):
    global _SAE_CACHE
    if layer in _SAE_CACHE:
        return _SAE_CACHE[layer]
    try:
        sae_model = load_sae_prot(ESM_DIM=1280, SAE_DIM=4096, LAYER=layer, device=get_device())
    except NameError as e:
        raise RuntimeError("load_sae_prot(...) not found. Edit the import block near the top to point to your module.") from e
    _SAE_CACHE[layer] = sae_model
    return sae_model

def get_esm():
    # Uses your predefined loader
    global _ESM_CACHE
    if _ESM_CACHE is not None:
        return _ESM_CACHE
    try:
        _ESM_CACHE = load_esm(33, device=get_device())
    except NameError as e:
        raise RuntimeError("load_esm(...) not found. Edit the import block near the top to point to your module.") from e
    return _ESM_CACHE

def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

def compute_latent_activations_on_sequence(
    sequence: str,
    layer: int,
    latent_idx: int,
    use_masked_latents: bool = True,
    use_error: bool = False,
) -> np.ndarray:
    """Runs your SAE hook at the target layer and returns a 1D vector (length L) for the chosen latent."""
    esm_transformer, batch_converter, esm2_alphabet = get_esm()
    device = get_device()
    # tokenization
    _, _, batch_tokens_BL = batch_converter([(1, sequence)])
    batch_tokens_BL = batch_tokens_BL.to(device)
    batch_mask_BL = (batch_tokens_BL != esm2_alphabet.padding_idx).to(device)
    cache_latents = True
    if use_masked_latents:
        cache_latents = False
    # SAE hook
    sae_model = get_sae(layer)
    try:
        hook = SAEHookProt(
            sae=sae_model,
            mask_BL=batch_mask_BL,
            cache_latents=cache_latents,
            cache_masked_latents=use_masked_latents,
            layer_is_lm=False,
            calc_error=False,
            use_error=use_error,
        )
    except NameError as e:
        raise RuntimeError("SAEHookProt not found. Edit imports at the top to your module.") from e
    handle = esm_transformer.esm.encoder.layer[layer].register_forward_hook(hook)
    with torch.no_grad():
        _ = esm_transformer.predict_contacts(batch_tokens_BL, batch_mask_BL)[0]
    handle.remove()
    # choose source array
    arr = None
    if use_error and hasattr(sae_model, "error_term") and sae_model.error_term is not None:
        arr = sae_model.error_term  # shape [B, L, F]
        arr = np.asarray(arr)[0]  # [L, F]
        # print(f"error_term shape: {arr.shape}")
    elif use_masked_latents and hasattr(sae_model, "masked_latents") and sae_model.masked_latents is not None:
        arr = sae_model.masked_latents  # [L, F] or [B, L, F]
        # print(f"masked_latents shape: {arr.shape}")
    elif cache_latents and hasattr(sae_model, "feature_acts") and sae_model.feature_acts is not None:
        arr = sae_model.feature_acts  # [L, F] or [B, L, F]

    arr = np.asarray(arr.cpu())
    if arr.ndim == 3:
        arr = arr[0]
    L, F = arr.shape
    if not (0 <= latent_idx < F):
        raise ValueError(f"latent_idx {latent_idx} out of range [0, {F-1}]")
    return arr[:, latent_idx]

def fetch_interpro_protein(acc: str, page_size: int = 500) -> dict:
    # All database entries (Pfam + others) mapped to this UniProt accession
    # url = f"https://www.ebi.ac.uk/interpro/api/protein/uniprot/{acc}" https://www.ebi.ac.uk/interpro/api/entry/interpro/protein/uniprot/I1RF61
    url = f"https://www.ebi.ac.uk/interpro/api/entry/interpro/protein/uniprot/{acc}"
    r = requests.get(url, timeout=30) #r = requests.get(url, params={"page_size": page_size}, timeout=30)
    r.raise_for_status()
    return r.json()

def parse_interpro_domains(interpro_json: dict) -> pd.DataFrame:
    rows = []
    for rec in interpro_json.get("results", []):
        meta = rec.get("metadata", {}) or {}
        entry = meta.get("accession")
        name = meta.get("name")
        db = meta.get("source_database")

        # NEW schema: entry → proteins[] → entry_protein_locations[] → fragments[]
        if "proteins" in rec:
            for prot in rec.get("proteins") or []:
                for loc in prot.get("entry_protein_locations") or []:
                    for frag in loc.get("fragments") or []:
                        rows.append({
                            "entry": entry,
                            "name": name,
                            "db": db,
                            "start": int(frag.get("start")) if frag.get("start") is not None else None,
                            "end": int(frag.get("end")) if frag.get("end") is not None else None,
                        })
        # OLD schema fallback (if you ever switch back)
        else:
            for loc in rec.get("entry_protein_locations", []) or []:
                for frag in loc.get("fragments") or []:
                    rows.append({
                        "entry": entry,
                        "name": name,
                        "db": db,
                        "start": int(frag.get("start")) if frag.get("start") is not None else None,
                        "end": int(frag.get("end")) if frag.get("end") is not None else None,
                    })

    if not rows:
        return pd.DataFrame(columns=["entry", "name", "db", "start", "end"])
    df = pd.DataFrame(rows).sort_values(["start", "end"]).reset_index(drop=True)
    return df




In [37]:
import time
import math
import numpy as np
import pandas as pd
from typing import Optional, Sequence, Tuple
from scipy.stats import fisher_exact, chi2

# --- small utils ---
def haldane_or(a,b,c,d):
    """Odds ratio with Haldane–Anscombe correction if any zero cell."""
    if min(a,b,c,d) == 0:
        a,b,c,d = a+0.5, b+0.5, c+0.5, d+0.5
    return (a*d)/(b*c)

def mantel_haenszel_or(tables):
    """
    Mantel–Haenszel common OR and 95% CI across strata (proteins).
    tables: iterable of (a,b,c,d) with all counts >= 0.
    Returns: OR_MH, (lo, hi), CMH_chi2, CMH_p
    """
    # MH estimator (fixed effects)
    sum_ad_n = 0.0
    sum_bc_n = 0.0
    sum_var_num = 0.0
    sum_var_den = 0.0

    for (a,b,c,d) in tables:
        n = a+b+c+d
        if n == 0: 
            continue
        sum_ad_n += (a*d)/n
        sum_bc_n += (b*c)/n
        # Variance pieces for log OR_MH (Robins, Greenland & Breslow, 1986 approx)
        # Avoid division by zero with tiny continuity
        an = a/n; bn = b/n; cn = c/n; dn = d/n
        p1 = (a+b)/n; p0 = (c+d)/n
        q1 = (a+c)/n; q0 = (b+d)/n
        # var(log OR_MH) ≈ sum[(p1*p0)/(q1*q0*n)]  (safe approx for large n)
        # Use a more stable form below
        eps = 1e-12
        sum_var_num += ( (a+b)*(c+d) ) / max(n-1, 1)
        sum_var_den += ( (a+c)*(b+d) ) / max(n-1, 1) + eps

    if sum_bc_n == 0:
        or_mh = np.inf
    else:
        or_mh = sum_ad_n / sum_bc_n

    # CMH test statistic (one df)
    # Q_MH = ( (Σ (a - E[a]))^2 ) / Σ Var[a], but use standard closed-form:
    # Here use log(OR_MH) variance approx for CI and CMH chi2 via
    #      (|Σ (a*d/n - b*c/n)|)^2 / Σ V_i
    # A simple, robust approximate variance for log(OR_MH):
    var_log_or = sum_var_num / (sum_var_den + 1e-12)
    se = math.sqrt(max(var_log_or, 1e-12))
    lo = math.exp(math.log(or_mh + 1e-300) - 1.96*se)
    hi = math.exp(math.log(or_mh + 1e-300) + 1.96*se)

    # CMH chi^2 via Woolf log-OR test
    # Test H0: log OR_MH = 0
    cmh_chi2 = (math.log(or_mh + 1e-300)**2) / (var_log_or + 1e-12)
    cmh_p = 1.0 - chi2.cdf(cmh_chi2, df=1)
    return or_mh, (lo, hi), cmh_chi2, cmh_p

def fishers_method(pvals):
    """Combine independent p-values (two-sided Fisher)."""
    pvals = np.asarray([p for p in pvals if np.isfinite(p) and p>0], dtype=float)
    if pvals.size == 0:
        return np.nan
    stat = -2.0 * np.sum(np.log(pvals))
    return 1.0 - chi2.cdf(stat, df=2*pvals.size)

# --- residue mask utilities ---
def top_fraction_mask(values: np.ndarray, frac: float = 0.05) -> np.ndarray:
    """Boolean mask for top frac of values (ties included)."""
    v = np.asarray(values, dtype=float).ravel()
    if v.size == 0:
        return np.zeros(0, dtype=bool)
    thr = np.quantile(v, 1.0 - frac)
    return v >= thr

def spans_to_mask(L: int, spans_df: pd.DataFrame, wanted_entries: set[str]) -> np.ndarray:
    """
    Build a boolean mask of length L for residues covered by any span whose 'entry'
    is in wanted_entries (InterPro accessions like 'IPR001234').
    The spans_df must have columns: entry, start, end (1-based inclusive).
    """
    mask = np.zeros(L, dtype=bool)
    if spans_df is None or spans_df.empty:
        return mask
    for _, row in spans_df.iterrows():
        if row.get("entry") in wanted_entries and row.get("start") is not None and row.get("end") is not None:
            s = int(row["start"]) - 1
            e = int(row["end"])     # slice end exclusive
            s = max(0, min(s, L))
            e = max(0, min(e, L))
            if e > s:
                mask[s:e] = True
    return mask

# --- core per-protein test ---
def residue_overlap_table(activations_1d: np.ndarray, span_mask: np.ndarray, top_frac: float = 0.05):
    """
    Make 2×2 for residues: active vs inactive (by top_frac) × in-span vs out-of-span.
      a: active & in-span
      b: active & out-of-span
      c: inactive & in-span
      d: inactive & out-of-span
    Returns (a,b,c,d), OR (Haldane), Fisher p (one-sided).
    """
    act_mask = top_fraction_mask(activations_1d, frac=top_frac)
    L = act_mask.size
    if span_mask.size != L:
        raise ValueError(f"Span mask length {span_mask.size} != activations length {L}")
    a = int(np.sum(act_mask & span_mask))
    b = int(np.sum(act_mask & ~span_mask))
    c = int(np.sum(~act_mask & span_mask))
    d = int(np.sum(~act_mask & ~span_mask))
    OR = haldane_or(a,b,c,d)
    _, p = fisher_exact([[a,b],[c,d]], alternative="greater")
    return (a,b,c,d), OR, p

# --- main: domain-overlap analysis for one latent & one InterPro domain ---
def latent_domain_span_enrichment(
    tmp,                            # your cached activations loader (has .metadata and .load_protein_activations_original_format)
    all_properties: np.ndarray,     # [N, M] domain indicators aligned to tmp.metadata['proteins']
    domain_col: int,                # column index in all_properties for your target domain
    layer: int, latent_idx: int,    # which latent to evaluate
    interpro_accession: str,        # e.g., "IPR016040"
    sample_in: int = 50,            # #proteins from domain to sample (cap)
    sample_out: int = 50,           # matched non-domain controls (cap)
    top_frac: float = 0.05,         # top residues as "active window"
    sleep_between_calls: float = 0.1,  # to be gentle with APIs
    entry_names: Optional[Sequence[str]] = None,   # optional mapping for logging
) -> Tuple[pd.DataFrame, dict]:
    """
    Returns:
      per_protein_df: one row per protein with 2x2, OR, Fisher p, lengths, counts
      summary: dict with MH OR, CI, CMH p, Fisher-combined p, and bookkeeping
    """
    D = (np.asarray(all_properties) != 0)
    in_mask = D[:, domain_col]
    N = D.shape[0]

    # protein indices
    idx_in = np.where(in_mask)[0]
    idx_out = np.where(~in_mask)[0]
    if idx_in.size == 0:
        raise ValueError("No proteins annotated for the requested domain_col.")
    # sample
    rng = np.random.default_rng(42)
    pick_in = rng.choice(idx_in, size=min(sample_in, idx_in.size), replace=False)
    pick_out = rng.choice(idx_out, size=min(sample_out, idx_out.size), replace=False)

    # iterate
    rows = []
    fisher_ps = []
    tables = []
    for subset, label in [(pick_in, "in"), (pick_out, "out")]:
        for pi in subset:
            prot = tmp.load_protein_activations_original_format(int(pi))
            acc = prot.get("protein_id")
            actsL = prot["activations"][layer].cpu().numpy()  # [L, F]
            L, F = actsL.shape
            if not (0 <= latent_idx < F):
                # skip if latent out of range for this layer dump
                continue
            lat_vec = actsL[:, latent_idx]  # [L]
            uniprot_acc = acc.split("|")[1]
            # fetch InterPro and build span mask for *this* domain
            try:
                interpro_json = fetch_interpro_protein(uniprot_acc)
            except Exception as e:
                # network hiccup; skip with a note
                rows.append({
                    "protein_ix": int(pi), "acc": acc, "set": label,
                    "L": int(L), "error": f"InterPro fetch failed: {e}"
                })
                continue
            spans_df = parse_interpro_domains(interpro_json)
            span_mask = spans_to_mask(L, spans_df, {interpro_accession})

            # 2x2 across residues
            (a,b,c,d), OR, p = residue_overlap_table(lat_vec, span_mask, top_frac=top_frac)
            fisher_ps.append(p)
            tables.append((a,b,c,d))
            rows.append({
                "protein_ix": int(pi),
                "acc": uniprot_acc,
                "set": label,
                "L": int(L),
                "n_active": int(top_fraction_mask(lat_vec, top_frac).sum()),
                "n_span": int(span_mask.sum()),
                "a": a, "b": b, "c": c, "d": d,
                "or_residue": OR,
                "fisher_p_residue": p,
                "error": None
            })
            time.sleep(sleep_between_calls)

    per_protein_df = pd.DataFrame(rows)

    # summarize across proteins
    if len(tables) == 0:
        summary = {"error": "no valid tables", "n_proteins": 0}
        return per_protein_df, summary

    or_mh, (ci_lo, ci_hi), cmh_chi2, cmh_p = mantel_haenszel_or(tables)
    p_fisher_combined = fishers_method(fisher_ps)

    summary = {
        "layer": layer,
        "latent_idx": latent_idx,
        "domain_col": int(domain_col),
        "interpro_accession": interpro_accession,
        "n_proteins": len(tables),
        "n_in_sampled": int(len(pick_in)),
        "n_out_sampled": int(len(pick_out)),
        "top_frac": top_frac,
        "OR_MH": or_mh,
        "OR_MH_CI95_lo": ci_lo,
        "OR_MH_CI95_hi": ci_hi,
        "CMH_chi2": cmh_chi2,
        "CMH_p": cmh_p,
        "Fisher_combined_p": p_fisher_combined,
    }
    return per_protein_df, summary


In [38]:
import time
import math
import numpy as np
import pandas as pd
from typing import Optional, Sequence, Tuple, Iterable
from scipy.stats import fisher_exact, chi2

# =========================
# Stats helpers
# =========================
def haldane_or(a,b,c,d):
    """Odds ratio with Haldane–Anscombe correction if any zero cell."""
    if min(a,b,c,d) == 0:
        a,b,c,d = a+0.5, b+0.5, c+0.5, d+0.5
    return (a*d)/(b*c)

def mantel_haenszel_or(tables: Iterable[Tuple[int,int,int,int]]):
    """
    Mantel–Haenszel common OR and 95% CI across strata (proteins).
    Returns: OR_MH, (lo, hi), CMH_chi2, CMH_p
    """
    tables = list(tables)
    if len(tables) == 0:
        return np.nan, (np.nan, np.nan), np.nan, np.nan

    sum_ad_n = 0.0
    sum_bc_n = 0.0
    sum_var_num = 0.0
    sum_var_den = 0.0

    for (a,b,c,d) in tables:
        n = a+b+c+d
        if n <= 1:
            continue
        sum_ad_n += (a*d)/n
        sum_bc_n += (b*c)/n
        # simple, stable variance pieces for log(OR_MH)
        sum_var_num += ((a+b)*(c+d)) / (n-1)
        sum_var_den += ((a+c)*(b+d)) / (n-1) + 1e-12

    or_mh = np.inf if sum_bc_n == 0 else (sum_ad_n / sum_bc_n)
    var_log_or = sum_var_num / (sum_var_den + 1e-12)
    se = math.sqrt(max(var_log_or, 1e-12))
    lo = math.exp(math.log(or_mh + 1e-300) - 1.96*se)
    hi = math.exp(math.log(or_mh + 1e-300) + 1.96*se)

    # CMH chi^2 (test H0: log OR_MH = 0)
    cmh_chi2 = (math.log(or_mh + 1e-300)**2) / (var_log_or + 1e-12)
    cmh_p = 1.0 - chi2.cdf(cmh_chi2, df=1)
    return or_mh, (lo, hi), cmh_chi2, cmh_p

def fishers_method(pvals):
    """Combine independent p-values (two-sided Fisher)."""
    pvals = np.asarray([p for p in pvals if np.isfinite(p) and p>0], dtype=float)
    if pvals.size == 0:
        return np.nan
    stat = -2.0 * np.sum(np.log(pvals))
    return 1.0 - chi2.cdf(stat, df=2*pvals.size)

# =========================
# Masks & spans
# =========================
def top_k_mask_exact(values: np.ndarray, frac: float = 0.05, rng_seed: int = 0) -> np.ndarray:
    """
    Pick exactly ceil(frac*L) residues as 'active', resolving ties stably.
    """
    v = np.asarray(values, float).ravel()
    L = v.size
    if L == 0:
        return np.zeros(0, bool)
    K = max(1, int(math.ceil(frac * L)))
    # get indices of top-K
    idx = np.argpartition(v, L-K)[L-K:]
    mask = np.zeros(L, bool)
    mask[idx] = True
    # tie resolution if more than K selected (rare but possible)
    if mask.sum() > K:
        rng = np.random.default_rng(rng_seed)
        boundary_val = v[idx].min()
        above = np.where(v > boundary_val)[0]
        on_boundary = np.where((v == boundary_val) & (~np.isin(np.arange(L), above)))[0]
        need = K - above.size
        if need < 0:
            # extremely many strictly-above due to numerical oddity: trim randomly
            keep = rng.choice(above, size=K, replace=False)
            mask[:] = False
            mask[keep] = True
        else:
            pick = rng.choice(on_boundary, size=max(0, need), replace=False)
            keep = np.concatenate([above, pick])
            mask[:] = False
            mask[keep] = True
    return mask

def threshold_mask(values: np.ndarray, thresh: float, zscore: bool = False) -> np.ndarray:
    """
    Threshold-based active mask.
    If zscore=True, compute z and apply z>thresh; else use raw values >= thresh.
    """
    v = np.asarray(values, float).ravel()
    if v.size == 0:
        return np.zeros(0, bool)
    if zscore:
        mu = float(v.mean())
        sd = float(v.std(ddof=0))
        if sd == 0.0:
            return np.zeros(v.size, bool)  # nothing stands out
        z = (v - mu) / sd
        return z > thresh
    return v >= thresh

def spans_to_mask(L: int, spans_df: pd.DataFrame, wanted_entries: set[str]) -> np.ndarray:
    """
    Boolean mask of residues covered by any InterPro 'entry' in wanted_entries.
    'start'/'end' are 1-based inclusive in InterPro; we convert to 0-based half-open.
    """
    mask = np.zeros(L, dtype=bool)
    if spans_df is None or spans_df.empty:
        return mask
    for _, row in spans_df.iterrows():
        if row.get("entry") in wanted_entries and row.get("start") is not None and row.get("end") is not None:
            s = int(row["start"]) - 1
            e = int(row["end"])
            s = max(0, min(s, L))
            e = max(0, min(e, L))
            if e > s:
                mask[s:e] = True
    return mask

# =========================
# Per-protein residue 2x2
# =========================
def residue_overlap_table(
    activations_1d: np.ndarray,
    span_mask: np.ndarray,
    mode: str = "top_frac",
    top_frac: float = 0.05,
    abs_thresh: Optional[float] = None,
    z_thresh: Optional[float] = None,
    rng_seed: int = 0,
):
    """
    Build 2×2 over residues: active vs inactive  × in-span vs out-of-span.
    mode: "top_frac" (default), "abs_thresh", or "z_thresh".
    Returns: (a,b,c,d), OR (Haldane), Fisher p (one-sided). May return (..), (nan, nan) if degenerate.
    """
    v = np.asarray(activations_1d, float).ravel()
    L = v.size
    if span_mask.size != L:
        raise ValueError(f"Span mask length {span_mask.size} != activations length {L}")

    if mode == "top_frac":
        act_mask = top_k_mask_exact(v, frac=top_frac, rng_seed=rng_seed)
    elif mode == "abs_thresh":
        if abs_thresh is None:
            raise ValueError("abs_thresh must be provided for mode='abs_thresh'")
        act_mask = threshold_mask(v, abs_thresh, zscore=False)
    elif mode == "z_thresh":
        if z_thresh is None:
            raise ValueError("z_thresh must be provided for mode='z_thresh'")
        act_mask = threshold_mask(v, z_thresh, zscore=True)
    else:
        raise ValueError("mode must be one of {'top_frac','abs_thresh','z_thresh'}")

    # counts
    a = int(np.sum(act_mask & span_mask))
    b = int(np.sum(act_mask & ~span_mask))
    c = int(np.sum(~act_mask & span_mask))
    d = int(np.sum(~act_mask & ~span_mask))

    # drop degenerate tables (any margin zero) -> no enrichment test possible
    if (a+b)==0 or (c+d)==0 or (a+c)==0 or (b+d)==0:
        return (a,b,c,d), np.nan, np.nan

    OR = haldane_or(a,b,c,d)
    _, p = fisher_exact([[a,b],[c,d]], alternative="greater")
    return (a,b,c,d), OR, p

# =========================
# Main: IN-DOMAIN ONLY
# =========================
def latent_domain_span_enrichment_inonly(
    tmp,                            # your cached activations loader
    all_properties: np.ndarray,     # [N, M] binary indicators
    domain_col: int,                # column in all_properties for target domain
    layer: int, latent_idx: int,    # latent to evaluate
    interpro_accession: str,        # e.g., "IPR016040"
    sample_in: int = 50,            # cap on in-domain proteins
    mode: str = "top_frac",         # "top_frac" | "abs_thresh" | "z_thresh"
    top_frac: float = 0.05,         # used if mode="top_frac"
    abs_thresh: Optional[float] = None,  # if mode="abs_thresh"
    z_thresh: Optional[float] = None,    # if mode="z_thresh"
    sleep_between_calls: float = 0.05,   # be nice to API
) -> Tuple[pd.DataFrame, dict]:
    """
    Samples ONLY in-domain proteins that actually have spans for `interpro_accession`.
    Returns:
      per_protein_df: rows for each protein kept
      summary: pooled MH OR, CI, CMH p, Fisher-combined p, and counts
    """
    D = (np.asarray(all_properties) != 0)
    in_mask = D[:, domain_col]
    idx_in = np.where(in_mask)[0]
    if idx_in.size == 0:
        raise ValueError("No proteins annotated for the requested domain_col.")

    rng = np.random.default_rng(42)
    pick_in = rng.choice(idx_in, size=min(sample_in, idx_in.size), replace=False)

    rows = []
    tables_valid = []
    fisher_ps_valid = []
    kept = 0
    skipped_nospan = 0
    skipped_degenerate = 0
    skipped_range = 0
    api_fail = 0

    for pi in pick_in:
        prot = tmp.load_protein_activations_original_format(int(pi))
        acc = prot.get("protein_id")
        # UniProt accession normalization (e.g., 'sp|Q9XXXX|NAME' -> 'Q9XXXX')
        uniprot_acc = acc.split("|")[1] if "|" in acc else acc

        actsL = prot["activations"][layer].cpu().numpy()  # [L, F]
        L, F = actsL.shape
        if not (0 <= latent_idx < F):
            skipped_range += 1
            continue
        lat_vec = actsL[:, latent_idx]

        # fetch InterPro & build spans for THIS domain only
        try:
            interpro_json = fetch_interpro_protein(uniprot_acc)
        except Exception as e:
            rows.append({
                "protein_ix": int(pi), "acc": uniprot_acc, "L": int(L),
                "n_active": np.nan, "n_span": np.nan, "a": np.nan, "b": np.nan, "c": np.nan, "d": np.nan,
                "or_residue": np.nan, "fisher_p_residue": np.nan,
                "error": f"InterPro fetch failed: {e}"
            })
            api_fail += 1
            continue

        spans_df = parse_interpro_domains(interpro_json)
        span_mask = spans_to_mask(L, spans_df, {interpro_accession})

        n_span = int(span_mask.sum())
        if n_span == 0 or n_span == L:
            # no informative partition; skip from pooled stats (still log it)
            rows.append({
                "protein_ix": int(pi), "acc": uniprot_acc, "L": int(L),
                "n_active": np.nan, "n_span": n_span, "a": np.nan, "b": np.nan, "c": np.nan, "d": np.nan,
                "or_residue": np.nan, "fisher_p_residue": np.nan,
                "error": "no spans or spans cover entire sequence"
            })
            skipped_nospan += 1
            time.sleep(sleep_between_calls)
            continue

        # build 2x2
        (a,b,c,d), OR, p = residue_overlap_table(
            lat_vec, span_mask,
            mode=mode, top_frac=top_frac, abs_thresh=abs_thresh, z_thresh=z_thresh
        )

        # count actives for logging (based on chosen mode)
        if mode == "top_frac":
            n_active = int(top_k_mask_exact(lat_vec, frac=top_frac).sum())
        elif mode == "abs_thresh":
            n_active = int(threshold_mask(lat_vec, abs_thresh, zscore=False).sum())
        else:
            n_active = int(threshold_mask(lat_vec, z_thresh, zscore=True).sum())

        row = {
            "protein_ix": int(pi),
            "acc": uniprot_acc,
            "L": int(L),
            "n_active": n_active,
            "n_span": n_span,
            "a": a, "b": b, "c": c, "d": d,
            "or_residue": OR,
            "fisher_p_residue": p,
            "error": None
        }

        # keep only non-degenerate tables for pooling
        if np.isnan(OR) or np.isnan(p):
            row["error"] = "degenerate 2x2 (no margins)"
            skipped_degenerate += 1
        else:
            tables_valid.append((a,b,c,d))
            fisher_ps_valid.append(p)
            kept += 1

        rows.append(row)
        time.sleep(sleep_between_calls)

    per_protein_df = pd.DataFrame(rows)

    # summarize across valid proteins
    if len(tables_valid) == 0:
        summary = {
            "layer": layer, "latent_idx": latent_idx, "domain_col": int(domain_col),
            "interpro_accession": interpro_accession,
            "mode": mode, "top_frac": top_frac, "abs_thresh": abs_thresh, "z_thresh": z_thresh,
            "n_in_sampled": int(len(pick_in)),
            "n_valid": 0, "n_skipped_nospan_or_allspan": int(skipped_nospan),
            "n_skipped_degenerate": int(skipped_degenerate),
            "n_skipped_range": int(skipped_range), "api_fail": int(api_fail),
            "OR_MH": np.nan, "OR_MH_CI95_lo": np.nan, "OR_MH_CI95_hi": np.nan,
            "CMH_chi2": np.nan, "CMH_p": np.nan, "Fisher_combined_p": np.nan
        }
        return per_protein_df, summary

    or_mh, (ci_lo, ci_hi), cmh_chi2, cmh_p = mantel_haenszel_or(tables_valid)
    p_fisher_combined = fishers_method(fisher_ps_valid)

    summary = {
        "layer": layer, "latent_idx": latent_idx, "domain_col": int(domain_col),
        "interpro_accession": interpro_accession,
        "mode": mode, "top_frac": top_frac, "abs_thresh": abs_thresh, "z_thresh": z_thresh,
        "n_in_sampled": int(len(pick_in)),
        "n_valid": int(len(tables_valid)),
        "n_skipped_nospan_or_allspan": int(skipped_nospan),
        "n_skipped_degenerate": int(skipped_degenerate),
        "n_skipped_range": int(skipped_range),
        "api_fail": int(api_fail),
        "OR_MH": or_mh,
        "OR_MH_CI95_lo": ci_lo,
        "OR_MH_CI95_hi": ci_hi,
        "CMH_chi2": cmh_chi2,
        "CMH_p": cmh_p,
        "Fisher_combined_p": p_fisher_combined
    }
    return per_protein_df, summary


In [39]:
from utils import load_cached_activations
tmp = load_cached_activations("/project/pi_annagreen_umass_edu/jatin/plm_circuits/acts")

Loaded cache with 10000 proteins


In [18]:
domain_col = 10047 #9059           # column in all_properties
ipr = "IPR001680" # "IPR015943" # "IPR029058"           # matching InterPro accession
layer = 12
latent_idx = 3035

per_prot, summ = latent_domain_span_enrichment_inonly(
    tmp=tmp,
    all_properties=all_properties,
    domain_col=domain_col,
    layer=layer,
    latent_idx=latent_idx,
    interpro_accession=ipr,
    sample_in=50,
    mode="abs_thresh",         # or "z_thresh", "abs_thresh"
    # top_frac=0.05,           # if mode="top_frac"
    abs_thresh=0.5,        # if mode="abs_thresh"
    # z_thresh=...,          # if mode="z_thresh"
    sleep_between_calls=0.05
)

per_prot.head(10)

,protein_ix,acc,L,n_active,n_span,a,b,c,d,or_residue,fisher_p_residue,error
0,8572,Q6CU55,441,83.0,291.0,79.0,4.0,212.0,146.0,13.601415,4.425464e-12,None
1,7848,Q8BGF3,359,71.0,214.0,57.0,14.0,157.0,131.0,3.397179,3.783532e-05,None
2,75,Q1E6Q0,527,79.0,164.0,52.0,27.0,112.0,336.0,5.777778,4.447238e-12,None
3,3007,Q8AVT9,372,51.0,204.0,37.0,14.0,167.0,154.0,2.437126,4.327698e-03,None
4,987,Q9Y297,607,72.0,287.0,68.0,4.0,219.0,316.0,24.529680,8.696362e-20,None
5,2584,P11017,342,111.0,291.0,109.0,2.0,182.0,49.0,14.673077,1.328904e-07,None
6,550,Q9LT47,371,82.0,244.0,73.0,9.0,171.0,118.0,5.597141,8.517630e-08,None
7,7740,Q9SU78,489,65.0,216.0,54.0,11.0,162.0,262.0,7.939394,5.721520e-12,None
8,3524,Q94AD8,541,51.0,281.0,46.0,5.0,235.0,255.0,9.982979,1.211086e-09,None
9,6524,Q8N0X2,633,111.0,288.0,111.0,0.0,177.0,345.0,434.064789,6.501279e-45,None


In [19]:
per_prot.head(30)

,protein_ix,acc,L,n_active,n_span,a,b,c,d,or_residue,fisher_p_residue,error
0,8572,Q6CU55,441,83.0,291.0,79.0,4.0,212.0,146.0,13.601415,4.425464e-12,None
1,7848,Q8BGF3,359,71.0,214.0,57.0,14.0,157.0,131.0,3.397179,3.783532e-05,None
2,75,Q1E6Q0,527,79.0,164.0,52.0,27.0,112.0,336.0,5.777778,4.447238e-12,None
3,3007,Q8AVT9,372,51.0,204.0,37.0,14.0,167.0,154.0,2.437126,4.327698e-03,None
4,987,Q9Y297,607,72.0,287.0,68.0,4.0,219.0,316.0,24.529680,8.696362e-20,None
5,2584,P11017,342,111.0,291.0,109.0,2.0,182.0,49.0,14.673077,1.328904e-07,None
6,550,Q9LT47,371,82.0,244.0,73.0,9.0,171.0,118.0,5.597141,8.517630e-08,None
7,7740,Q9SU78,489,65.0,216.0,54.0,11.0,162.0,262.0,7.939394,5.721520e-12,None
8,3524,Q94AD8,541,51.0,281.0,46.0,5.0,235.0,255.0,9.982979,1.211086e-09,None
9,6524,Q8N0X2,633,111.0,288.0,111.0,0.0,177.0,345.0,434.064789,6.501279e-45,None


In [22]:
pd.DataFrame([summ])

,layer,latent_idx,domain_col,interpro_accession,mode,top_frac,abs_thresh,z_thresh,n_in_sampled,n_valid,n_skipped_nospan_or_allspan,n_skipped_degenerate,n_skipped_range,api_fail,OR_MH,OR_MH_CI95_lo,OR_MH_CI95_hi,CMH_chi2,CMH_p,Fisher_combined_p
0,12,3035,10047,IPR001680,abs_thresh,0.05,0.5,None,43,42,0,0,0,1,6.254656,1.241059,31.522043,4.936028,0.026302,0.0


In [17]:
interpro_annotations_nonzero.iloc[10047]

ENTRY_AC        IPR001680
ENTRY_TYPE         Repeat
ENTRY_NAME    WD40 repeat
Name: 10047, dtype: object

In [22]:
domain_col = 9059   # example column idx in all_properties
ipr = "IPR029058"   # make sure this matches the same domain as column 9005
layer = 12
latent_idx = 2112

In [23]:
per_protein_df, summary = latent_domain_span_enrichment(
    tmp=tmp,
    all_properties=all_properties,
    domain_col=domain_col,
    layer=layer,
    latent_idx=latent_idx,
    interpro_accession=ipr,
    sample_in=50,
    sample_out=50,
    top_frac=0.05,           # top 5% residues per protein as the "activation window"
    sleep_between_calls=0.1,
)

print(pd.DataFrame([summary]))
display_cols = ["acc","set","L","n_active","n_span","a","b","c","d","or_residue","fisher_p_residue","error"]
print(per_protein_df[display_cols].head(20))

   layer  latent_idx  domain_col interpro_accession  n_proteins  n_in_sampled  \
0     12        2112        9059          IPR029058         100            50   

   n_out_sampled  top_frac      OR_MH  OR_MH_CI95_lo  OR_MH_CI95_hi  \
0             50      0.05  25.499074        7.47574      86.975034   

    CMH_chi2         CMH_p  Fisher_combined_p  
0  26.764788  2.297865e-07       4.952705e-13  
       acc set    L  n_active  n_span    a    b    c    d  or_residue  \
0   Q1RFI8  in  279        14     277   14    0  263    2    0.275142   
1   Q5XTQ4  in  576        29     544   29    0  515   32    3.719690   
2   C9SJ15  in  877       877     224  224  653    0    0    0.343535   
3   P0DMB4  in  336        17     329   17    0  312    7    0.840000   
4   C7YRS6  in  615        31     437   31    0  406  178   27.664207   
5   Q6Q252  in  339        17     301   17    0  284   38    4.736380   
6   C5FYZ3  in  921       921     266  266  655    0    0    0.406560   
7   Q6F9F4  in

In [25]:
per_protein_df[display_cols].head(100)

,acc,set,L,n_active,n_span,a,b,c,d,or_residue,fisher_p_residue,error
0,Q1RFI8,in,279,14,277,14,0,263,2,0.275142,0.901988,None
1,Q5XTQ4,in,576,29,544,29,0,515,32,3.719690,0.182588,None
2,C9SJ15,in,877,877,224,224,653,0,0,0.343535,1.000000,None
3,P0DMB4,in,336,17,329,17,0,312,7,0.840000,0.692937,None
4,C7YRS6,in,615,31,437,31,0,406,178,27.664207,0.000018,None
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Q6QDB6,out,482,25,0,0,25,0,457,17.941176,1.000000,None
96,P0DQN1,out,36,36,0,0,36,0,0,0.013699,1.000000,None
97,O23810,out,97,97,0,0,97,0,0,0.005128,1.000000,None
98,Q8Z2X8,out,342,342,0,0,342,0,0,0.001460,1.000000,None


In [26]:
pd.DataFrame([summary])

,layer,latent_idx,domain_col,interpro_accession,n_proteins,n_in_sampled,n_out_sampled,top_frac,OR_MH,OR_MH_CI95_lo,OR_MH_CI95_hi,CMH_chi2,CMH_p,Fisher_combined_p
0,12,2112,9059,IPR029058,100,50,50,0.05,25.499074,7.47574,86.975034,26.764788,2.297865e-07,4.952705e-13


In [ ]:
D = (np.asarray(all_properties) != 0)
in_mask = D[:, domain_col]
N = D.shape[0]
print(f"N: {N}")

# protein indices
idx_in = np.where(in_mask)[0]
idx_out = np.where(~in_mask)[0]

rng = np.random.default_rng(42)
pick_in = rng.choice(idx_in, size=min(50, idx_in.size), replace=False)

for pi in pick_in:
    prot = tmp.load_protein_activations_original_format(int(pi))
    acc = prot.get("protein_id")
    acts_LF = prot["activations"][layer].cpu().numpy()  # [L, F]
    L, F = acts_LF.shape
    
    lat_vec_L = acts_LF[:, latent_idx]
    uniprot_acc = acc.split("|")[1]
    # fetch InterPro and build span mask for *this* domain
    try:
        interpro_json = fetch_interpro_protein(acc)
    except Exception as e:
        # network hiccup; skip with a note
        rows.append({
            "protein_ix": int(pi), "acc": acc, "set": label,
            "L": int(L), "error": f"InterPro fetch failed: {e}"
        })
        continue
    spans_df = parse_interpro_domains(interpro_json)
    span_mask = spans_to_mask(L, spans_df, {interpro_accession})

    # 2x2 across residues
    (a,b,c,d), OR, p = residue_overlap_table(lat_vec, span_mask, top_frac=top_frac)
    fisher_ps.append(p)
    tables.append((a,b,c,d))
    rows.append({
        "protein_ix": int(pi),
        "acc": acc,
        "set": label,
        "L": int(L),
        "n_active": int(top_fraction_mask(lat_vec, top_frac).sum()),
        "n_span": int(span_mask.sum()),
        "a": a, "b": b, "c": c, "d": d,
        "or_residue": OR,
        "fisher_p_residue": p,
        "error": None
    })
    time.sleep(sleep_between_calls)
    break


N: 10000
{'count': 3, 'next': None, 'previous': None, 'results': [{'metadata': {'accession': 'IPR000801', 'name': 'Esterase-like', 'source_database': 'interpro', 'type': 'family', 'integrated': None, 'member_databases': {'pfam': {'PF00756': 'Putative esterase'}}, 'go_terms': None}, 'proteins': [{'accession': 'q1rfi8', 'protein_length': 277, 'source_database': 'reviewed', 'organism': '364106', 'in_alphafold': True, 'in_bfvd': False, 'entry_protein_locations': [{'fragments': [{'start': 21, 'end': 268, 'dc-status': 'CONTINUOUS'}], 'representative': False, 'model': None, 'score': None}]}]}, {'metadata': {'accession': 'IPR014186', 'name': 'S-formylglutathione hydrolase', 'source_database': 'interpro', 'type': 'family', 'integrated': None, 'member_databases': {'panther': {'PTHR10061': 'S-FORMYLGLUTATHIONE HYDROLASE'}, 'ncbifam': {'TIGR02821': 'S-formylglutathione hydrolase'}}, 'go_terms': [{'identifier': 'GO:0018738', 'name': 'S-formylglutathione hydrolase activity', 'category': {'code': 'F'

In [15]:


per_protein_df, summary = latent_domain_span_enrichment(
    tmp=tmp,
    all_properties=all_properties,
    domain_col=domain_col,
    layer=layer,
    latent_idx=latent_idx,
    interpro_accession=ipr,
    sample_in=50,
    sample_out=50,
    top_frac=0.05,           # top 5% residues per protein as the "activation window"
    sleep_between_calls=0.1,
)

print(pd.DataFrame([summary]))
display_cols = ["acc","set","L","n_active","n_span","a","b","c","d","or_residue","fisher_p_residue","error"]
print(per_protein_df[display_cols].head(20))

             error  n_proteins
0  no valid tables           0


KeyError: "['n_active', 'n_span', 'a', 'b', 'c', 'd', 'or_residue', 'fisher_p_residue'] not in index"

# Structure bits

In [42]:
import os
import requests
import pandas as pd
import mdtraj as md
from Bio.PDB import PDBParser  # only for pLDDT via B-factors

AF_URL_TPL = "https://alphafold.ebi.ac.uk/files/AF-{uniprot}-F1-model_v4.pdb"

# --- Your base mapping + sensible fallbacks ---
aa_3to1 = {
    "ALA": "A","CYS": "C","ASP": "D","GLU": "E","PHE": "F","GLY": "G",
    "HIS": "H","ILE": "I","LYS": "K","LEU": "L","MET": "M","ASN": "N",
    "PRO": "P","GLN": "Q","ARG": "R","SER": "S","THR": "T","VAL": "V",
    "TRP": "W","TYR": "Y",
    # fallbacks & modified residues you’ll often see
    "MSE": "M",  # selenomethionine
    "SEC": "U",  # selenocysteine
    "PYL": "O",  # pyrrolysine
    "ASX": "B",  # Asn/Asp ambiguous
    "GLX": "Z",  # Gln/Glu ambiguous
    "XLE": "J",  # Leu/Ile ambiguous
    "UNK": "X","XAA": "X","XXX": "X"
}

def three_to_one_safe(resname: str) -> str:
    # Normalize e.g. "MSE ", "mse", etc.
    key = (resname or "").strip().upper()
    return aa_3to1.get(key, "X")

def get_af_pdb_path(uniprot_id: str, pdb_dir: str = "pdbs") -> str:
    os.makedirs(pdb_dir, exist_ok=True)
    pdb_path = os.path.join(pdb_dir, f"AF-{uniprot_id}-F1-model_v4.pdb")
    if not os.path.exists(pdb_path):
        url = AF_URL_TPL.format(uniprot=uniprot_id)
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        with open(pdb_path, "w") as f:
            f.write(r.text)
    return pdb_path

def dssp_annotations_from_pdb(pdb_path: str, simplified: bool = True) -> pd.DataFrame:
    """
    Compute per-residue secondary structure with mdtraj (no mkdssp needed).
    simplified=True -> H/E/C; False -> H,B,E,G,I,T,S,' ' (8-state).
    Adds optional pLDDT (mean B-factor per residue) from AF PDB.
    """
    # mdtraj sometimes prints warnings on altlocs; safe to ignore
    traj = md.load_pdb(pdb_path)
    ss = md.compute_dssp(traj, simplified=simplified)[0]  # shape (n_res,)
    residues = list(traj.topology.residues)

    rows = []
    for i, res in enumerate(residues):
        rows.append({
            "chain_index": res.chain.index,          # integer chain index
            "resSeq": res.resSeq,                    # PDB residue number (can have gaps)
            "resName": res.name,                     # three-letter name
            "one_letter": three_to_one_safe(res.name),
            "ss": ss[i],                             # 'H','E','C' or 8-state char
            "residue_index_0based": i,
        })
    df = pd.DataFrame(rows)

    # Attach pLDDT (AlphaFold confidence is stored in B-factors)
    try:
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure("af", pdb_path)
        b_means = []
        for model in structure:
            for chain in model:
                for residue in chain:
                    bvals = [atom.get_bfactor() for atom in residue.get_atoms()]
                    b_means.append(sum(bvals)/len(bvals) if bvals else None)
        if len(b_means) >= len(df):
            df["plddt_mean"] = b_means[:len(df)]
    except Exception:
        # If Biopython hits an edge case, just continue without pLDDT.
        pass

    return df

def summarize_segments(df: pd.DataFrame) -> pd.DataFrame:
    """
    Collapse contiguous runs of the same SS per chain into segments.
    """
    segs = []
    for chain_idx, sub in df.groupby("chain_index", sort=False):
        sub = sub.reset_index(drop=True)
        if sub.empty:
            continue
        prev = sub.loc[0, "ss"]
        start = 0
        for i in range(1, len(sub)):
            cur = sub.loc[i, "ss"]
            if cur != prev:
                segs.append({
                    "chain_index": chain_idx,
                    "ss": prev,
                    "start_resSeq": int(sub.loc[start, "resSeq"]),
                    "end_resSeq": int(sub.loc[i-1, "resSeq"]),
                    "length": (i-1) - start + 1,
                })
                prev = cur
                start = i
        # close last
        segs.append({
            "chain_index": chain_idx,
            "ss": prev,
            "start_resSeq": int(sub.loc[start, "resSeq"]),
            "end_resSeq": int(sub.loc[len(sub)-1, "resSeq"]),
            "length": len(sub) - start,
        })
    return pd.DataFrame(segs).sort_values(["chain_index", "start_resSeq"])

def annotate_uniprot(uniprot_id: str, pdb_dir="pdbs", simplified=True):
    pdb_path = get_af_pdb_path(uniprot_id, pdb_dir=pdb_dir)
    df = dssp_annotations_from_pdb(pdb_path, simplified=simplified)
    segs = summarize_segments(df)
    return df, segs

# Example:
# df_residues, df_segments = annotate_uniprot("Q27743", simplified=True)
# df_residues.head(), df_segments.head()


In [43]:
from typing import Iterable, List, Optional, Union, Dict

def sample_uniprot_ids_for_domain_columns(
    tmp,
    all_properties: np.ndarray,
    interpro_annotations_nonzero: pd.DataFrame,
    domain_cols: Union[int, Iterable[int]],
    n: int = 10,
    seed: Optional[int] = None,
    allow_replacement: bool = False,
) -> List[Dict[str, object]]:
    """
    Sample proteins for one or more domain columns and report their UniProt IDs.

    Parameters
    ----------
    tmp : object
        Loader that exposes `load_protein_activations_original_format(idx)`.
    all_properties : np.ndarray
        Binary matrix [num_proteins, num_domains]; non-zero means the protein carries that domain.
    interpro_annotations_nonzero : pd.DataFrame
        Annotation table aligned with columns of `all_properties`, containing at least
        `ENTRY_AC` (accession) and optionally `ENTRY_NAME`.
    domain_cols : int or iterable of int
        Column index/indices into `all_properties`.
    n : int, optional
        Number of proteins to sample per domain (defaults to 10).
    seed : int, optional
        Seed for reproducible sampling. Each domain draws from the same RNG stream.
    allow_replacement : bool, optional
        If True and fewer proteins than requested are available, sample with replacement;
        otherwise the sample is capped at the number of available proteins.

    Returns
    -------
    List[dict]
        Each entry contains fields:
        `domain_col`, `interpro_accession`, `interpro_name`, `protein_idx`,
        `uniprot_id`, and `raw_id`.
    """
    domain_cols = [int(domain_cols)] if np.isscalar(domain_cols) else [int(c) for c in domain_cols]
    properties = np.asarray(all_properties)
    if properties.ndim != 2:
        raise ValueError("`all_properties` must be a 2-D array.")

    results: List[Dict[str, object]] = []
    rng = np.random.default_rng(seed)

    for col in domain_cols:
        if not (0 <= col < properties.shape[1]):
            raise IndexError(f"domain_col {col} out of range [0, {properties.shape[1]-1}].")

        row = interpro_annotations_nonzero.iloc[col]
        accession = row.get("ENTRY_AC", None)
        if pd.isna(accession):
            raise ValueError(f"No InterPro accession found for column {col} in annotations.")
        name = row.get("ENTRY_NAME", None)

        mask = properties[:, col] != 0
        idxs = np.flatnonzero(mask)
        if idxs.size == 0:
            raise ValueError(f"No proteins annotated for domain column {col} ({accession}).")

        replace = allow_replacement and (n > idxs.size)
        sample_size = n if replace else min(n, idxs.size)
        chosen = rng.choice(idxs, size=sample_size, replace=replace)

        for protein_idx in np.atleast_1d(chosen):
            prot = tmp.load_protein_activations_original_format(int(protein_idx))
            raw_id = prot.get("protein_id", "")
            uniprot_id = raw_id.split("|")[1] if isinstance(raw_id, str) and "|" in raw_id else raw_id
            results.append(
                {
                    "domain_col": col,
                    "interpro_accession": accession,
                    "interpro_name": name,
                    "protein_idx": int(protein_idx),
                    "uniprot_id": uniprot_id,
                    "raw_id": raw_id,
                }
            )

    return results


In [49]:
samples = sample_uniprot_ids_for_domain_columns(
    tmp,
    all_properties,
    interpro_annotations_nonzero,
    domain_cols=[9059, 8787],
    n=5,
    seed=123,
)
print(samples)

[{'domain_col': 9059, 'interpro_accession': 'IPR029058', 'interpro_name': 'Alpha/Beta hydrolase fold', 'protein_idx': 371, 'uniprot_id': 'Q75P26', 'raw_id': 'sp|Q75P26|AXE1_ASPOR'}, {'domain_col': 9059, 'interpro_accession': 'IPR029058', 'interpro_name': 'Alpha/Beta hydrolase fold', 'protein_idx': 7018, 'uniprot_id': 'Q0CNE3', 'raw_id': 'sp|Q0CNE3|CUTI3_ASPTN'}, {'domain_col': 9059, 'interpro_accession': 'IPR029058', 'interpro_name': 'Alpha/Beta hydrolase fold', 'protein_idx': 172, 'uniprot_id': 'P67367', 'raw_id': 'sp|P67367|YCFP_SALTI'}, {'domain_col': 9059, 'interpro_accession': 'IPR029058', 'interpro_name': 'Alpha/Beta hydrolase fold', 'protein_idx': 9190, 'uniprot_id': 'B2IY96', 'raw_id': 'sp|B2IY96|METXA_NOSP7'}, {'domain_col': 9059, 'interpro_accession': 'IPR029058', 'interpro_name': 'Alpha/Beta hydrolase fold', 'protein_idx': 7395, 'uniprot_id': 'P42840', 'raw_id': 'sp|P42840|YN60_YEAST'}, {'domain_col': 8787, 'interpro_accession': 'IPR015943', 'interpro_name': 'WD40/YVTN repea

In [50]:
layer = 12
latent_idx = 3035
for s in samples:
    print(f"Domain : {s['interpro_name']}")
    print(f"Protein: {s['uniprot_id']}")
    uniprot_acc = s['uniprot_id']
    df_residues, df_segments = annotate_uniprot(uniprot_acc, simplified=True)
    ss_string = ''.join(df_residues['ss'].values)
    seq = ''.join(df_residues['one_letter'].values)
    vals = compute_latent_activations_on_sequence(
        sequence=seq,
        layer=layer,
        latent_idx=latent_idx,
        use_masked_latents=True,
        use_error=False,
    )
    from IPython.display import display  # HTML import not needed for this call

    hm_html_obj_seq = render_sequence_heatmap(
        seq,
        vals[1:-1],
        title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
    )
    display(hm_html_obj_seq)

    hm_html_obj_ss = render_sequence_heatmap(
        ss_string,
        vals[1:-1],
        title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
    )
    display(hm_html_obj_ss)


Domain : Alpha/Beta hydrolase fold
Protein: Q75P26


/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


Domain : Alpha/Beta hydrolase fold
Protein: Q0CNE3


Domain : Alpha/Beta hydrolase fold
Protein: P67367


...

/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


Domain : WD40/YVTN repeat-like-containing domain superfamily
Protein: P26449


/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


Domain : WD40/YVTN repeat-like-containing domain superfamily
Protein: A5E5Y8


/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


In [32]:
uniprot_acc = "P45131"
df_residues, df_segments = annotate_uniprot(uniprot_acc, simplified=True)
ss_string = ''.join(df_residues['ss'].values)
seq = ''.join(df_residues['one_letter'].values)
vals = compute_latent_activations_on_sequence(
    sequence=seq,
    layer=12,
    latent_idx=3035,
    use_masked_latents=True,
    use_error=False,
)
from IPython.display import display  # HTML import not needed for this call

hm_html_obj_seq = render_sequence_heatmap(
    seq,
    vals[1:-1],
    title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
)
display(hm_html_obj_seq)

hm_html_obj_ss = render_sequence_heatmap(
    ss_string,
    vals[1:-1],
    title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
)
display(hm_html_obj_ss)


/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


In [33]:
uniprot_acc = "Q8QFR2"
df_residues, df_segments = annotate_uniprot(uniprot_acc, simplified=True)
ss_string = ''.join(df_residues['ss'].values)
seq = ''.join(df_residues['one_letter'].values)
vals = compute_latent_activations_on_sequence(
    sequence=seq,
    layer=12,
    latent_idx=3035,
    use_masked_latents=True,
    use_error=False,
)
from IPython.display import display  # HTML import not needed for this call

hm_html_obj_seq = render_sequence_heatmap(
    seq,
    vals[1:-1],
    title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
)
display(hm_html_obj_seq)

hm_html_obj_ss = render_sequence_heatmap(
    ss_string,
    vals[1:-1],
    title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
)
display(hm_html_obj_ss)


/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


In [30]:
len(vals)

482

In [31]:
len(seq)
len(ss_string)


480

In [34]:
uniprot_acc = "Q9UQ03"
df_residues, df_segments = annotate_uniprot(uniprot_acc, simplified=True)
ss_string = ''.join(df_residues['ss'].values)
seq = ''.join(df_residues['one_letter'].values)
vals = compute_latent_activations_on_sequence(
    sequence=seq,
    layer=12,
    latent_idx=3035,
    use_masked_latents=True,
    use_error=False,
)
from IPython.display import display  # HTML import not needed for this call

hm_html_obj_seq = render_sequence_heatmap(
    seq,
    vals[1:-1],
    title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
)
display(hm_html_obj_seq)

hm_html_obj_ss = render_sequence_heatmap(
    ss_string,
    vals[1:-1],
    title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
)
display(hm_html_obj_ss)

/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


In [21]:
df_residues, df_segments = annotate_uniprot("P45131", simplified=True)
df_residues.head()

/home/jnainani_umass_edu/.conda/envs/finetuning/lib/python3.10/site-packages/mdtraj/formats/pdb/pdbfile.py:208: UserWarning: Unlikely unit cell vectors detected in PDB file likely resulting from a dummy CRYST1 record. Discarding unit cell vectors.
  warnings.warn(


,chain_index,resSeq,resName,one_letter,ss,residue_index_0based,plddt_mean
0,0,1,MET,M,C,0,62.21
1,0,2,SER,S,C,1,82.92
2,0,3,VAL,V,C,2,95.59
3,0,4,GLN,Q,E,3,98.45
4,0,5,ASN,N,E,4,98.64


In [23]:
# Get combined string of secondary structure assignments
ss_string = ''.join(df_residues['ss'].values)
seq = ''.join(df_residues['one_letter'].values)
print(ss_string)
print(seq)


CCCEEEEECCCCCEECCCCCEECCEEEEEEEECCCCCCCCCEEEEECCCCCCCCCCCCCCCCCCCHHHEECCCCEECCCCEEEEECCCCCCCCCCCCCCECCCCCCECHHHCCCCCHHHHHHHHHHHHHHCCCCCEEEEEEECHHHHHHHHHHHHCCCCEEEEEEECCCCCCCHHHHHHHHHHHHHHHCCCCCHHHCCCCCCCCHHHHHHHHHHHHHHCCCHHHHHHHCCCCECCCCCCCCCCEHHHHHHHHHHHHHHCCCCHHHHHHHHHHHHCCCCCCCCCCHHHHHCCCCCEEEEEEECCCCCCCHHHHHHHHHHHHHCCCCEEEEEECCCCHHHHHHHCHHHHHHHHHHHHHCC
MSVQNVVLFDTQPLTLMLGGKLSHINVAYQTYGTLNAEKNNAVLICHALTGDAEPYFDDGRDGWWQNFMGAGLALDTDRYFFISSNVLGGCKGTTGPSSINPQTGKPYGSQFPNIVVQDIVKVQKALLDHLGISHLKAIIGGSFGGMQANQWAIDYPDFMDNIVNLCSSIYFSAEAIGFNHVMRQAVINDPNFNGGDYYEGTPPDQGLSIARMLGMLTYRTDLQLAKAFGRATKSDGSFWGDYFQVESYLSYQGKKFLERFDANSYLHLLRALDMYDPSLGYDNVKEALSRIKARYTLVSVTTDQLFKPIDLYKSKQLLEQSGVDLHFYEFPSDYGHDAFLVDYDQFEKRIRDGLAGN


In [46]:
from ui.html_gen import render_sequence_heatmap, compute_latent_activations_on_sequence

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
In IPython
Set autoreload


In [25]:
vals = compute_latent_activations_on_sequence(
    sequence=seq,
    layer=12,
    latent_idx=3035,
    use_masked_latents=True,
    use_error=False,
)

In [ ]:
from IPython.display import display  # HTML import not needed for this call

hm_html_obj = render_sequence_heatmap(
    seq,
    vals,
    title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
)
display(hm_html_obj)

In [26]:
from IPython.display import display  # HTML import not needed for this call

hm_html_obj = render_sequence_heatmap(
    ss_string,
    vals,
    title=f"Sequence heatmap for {uniprot_acc} (L{layer}-{latent_idx})",
)
display(hm_html_obj)

NameError: name 'uniprot_acc' is not defined

In [48]:
len(ss_string)

358

In [49]:
len(seq)

NameError: name 'seq' is not defined

In [46]:
df_segments[df_segments["ss"]=="E"].head(25)  # helix segments

,chain_index,ss,start_resSeq,end_resSeq,length
1,0,E,4,8,5
3,0,E,14,15,2
5,0,E,21,22,2
7,0,E,25,32,8
9,0,E,42,46,5
12,0,E,69,70,2
14,0,E,75,76,2
16,0,E,81,85,5
18,0,E,100,100,1
20,0,E,107,107,1


# Summary activation vs protein ranks 

# MWU test

# Tail Enrichment test